# LeetCode 273 — Integer to English Words (Đọc số nguyên thành chữ tiếng Anh) bằng Python

Notebook này giảng nhập môn bài **Hard**:

```python
class Solution:
    def numberToWords(self, num: int) -> str:

```

Đây là một bài **mô phỏng / chia nhỏ vấn đề** (simulation / decomposition): không cần cấu trúc dữ liệu cao siêu, không cần thuật toán phức tạp. Cái "Hard" nằm ở chỗ có **rất nhiều chi tiết nhỏ dễ sai**: số `0`, các số `11`–`19` có tên riêng, nhóm toàn số 0 nằm giữa (như `1,000,010`), khoảng trắng thừa giữa các từ... Bí quyết là **chia bài lớn thành các bài nhỏ**, mỗi bài nhỏ chỉ trả lời một câu hỏi, test riêng, rồi mới ghép lại.

**Mục tiêu của notebook:**

- Hiểu cách người Anh/Mỹ đọc số, **so sánh với cách đọc tiếng Việt** — bạn sẽ thấy hai ngôn ngữ giống nhau ở điểm quan trọng nhất: cùng chia số thành **nhóm 3 chữ số**.
- Học nền tảng Python từ đầu: list làm **bảng tra** (lookup table), `%` và `//`, `divmod`, vòng `while` tách nhóm, `str.join`, lọc chuỗi rỗng, **đệ quy** (recursion), `dict`.
- Biết **chia nhỏ**: đọc số `< 20` → `< 100` → `< 1000` → ghép theo nhóm `Thousand` / `Million` / `Billion`.
- Cài đặt và so sánh các cách:
  1. **Cách 1 (cách chính):** if/else rõ ràng cho từng tầng + helper `three_digits`, rồi ghép theo nhóm.
  2. **Cách 2:** đệ quy `helper(n)` theo các ngưỡng `20, 100, 1000, 10^6, 10^9` — gọn.
  3. **Cách 3:** bảng `(giá trị, từ)` duyệt từ lớn xuống — kiểu **greedy** giống đổi số sang số La Mã.
  4. **Cách 4 (chiều ngược):** đọc chữ → số (`words_to_number`) để kiểm chứng **round-trip**.
- Thuộc các edge case, nhận ra lỗi thường gặp, và tự kiểm thử bằng vét cạn + ngẫu nhiên có `random.seed`.

> Ghi chú: bài này **không** phải bài Premium — ai cũng nộp được trên LeetCode.

## 1. Đề bài

**Đề (dịch sát bản LeetCode):** Chuyển một số nguyên **không âm** `num` thành cách đọc bằng **chữ tiếng Anh** (English words representation) của nó.

**Ví dụ 1**

```text
Input:  num = 123
Output: "One Hundred Twenty Three"
```

**Ví dụ 2**

```text
Input:  num = 12345
Output: "Twelve Thousand Three Hundred Forty Five"
```

**Ví dụ 3**

```text
Input:  num = 1234567
Output: "One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven"
```

### Constraints

- `0 <= num <= 2^31 - 1` (tức là `num` không vượt quá `2,147,483,647` — khoảng 2,1 tỷ).

### Quy ước định dạng đầu ra (nhìn từ các ví dụ — LeetCode so khớp **từng ký tự**)

- Mỗi từ **viết hoa chữ cái đầu**: `"One Hundred"`, không phải `"one hundred"`.
- Giữa hai từ có **đúng 1 dấu cách**; không có dấu cách ở đầu hay cuối chuỗi.
- **Không** có chữ `"and"` (kiểu Anh-Anh hay nói "one hundred **and** five"), **không** có dấu gạch nối (`"Twenty-Three"`), **không** có dấu phẩy.
- Số `0` đọc là `"Zero"`.

Vì `num` lớn nhất cỡ 2,1 tỷ, từ chỉ **cấp** lớn nhất ta cần là `"Billion"` (tỷ).

In [1]:
examples = [
    (123, "One Hundred Twenty Three"),
    (12345, "Twelve Thousand Three Hundred Forty Five"),
    (1234567, "One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven"),
]

for num, expected in examples:
    print(f"{num:>8} -> {expected}")

MAX_NUM = 2**31 - 1
print()
print("Giới hạn trên của đề:", MAX_NUM)
print("Số chữ số tối đa    :", len(str(MAX_NUM)))

     123 -> One Hundred Twenty Three
   12345 -> Twelve Thousand Three Hundred Forty Five
 1234567 -> One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven

Giới hạn trên của đề: 2147483647
Số chữ số tối đa    : 10


### Quan sát Ví dụ 3 — "bản thiết kế" của thuật toán

Viết `1234567` theo kiểu có dấu phẩy ngăn nhóm: `1,234,567`. Đặt cạnh kết quả mong đợi:

| Phần của số | Giá trị nhóm | Đọc nhóm như một số nhỏ | Từ chỉ cấp |
|---|---:|---|---|
| **`1`**`,234,567` | 1 | `One` | `Million` |
| `1,`**`234`**`,567` | 234 | `Two Hundred Thirty Four` | `Thousand` |
| `1,234,`**`567`** | 567 | `Five Hundred Sixty Seven` | (không có) |

Ghép theo thứ tự từ trái sang phải: `One Million` + `Two Hundred Thirty Four Thousand` + `Five Hundred Sixty Seven`.

Phát hiện quan trọng: **mỗi nhóm chỉ là một số từ 0 đến 999**. Cụm `"Two Hundred Thirty Four"` xuất hiện y hệt khi đọc `234`, `234,000` hay `234,000,000` — chỉ khác từ chỉ cấp đứng sau. Vậy nếu ta viết được hàm đọc **mọi số từ 1 đến 999**, phần còn lại chỉ là "gắn đúng nhãn cấp". Đây cũng chính là gợi ý mà LeetCode đưa ra cho bài này: *nhóm số theo nghìn (3 chữ số), viết một helper đọc số nhỏ hơn 1000*.

## 2. Đọc số tiếng Anh — so sánh với tiếng Việt

Trước khi viết code, hãy hiểu **con người** đọc số thế nào. Đặt cách đọc tiếng Việt cạnh tiếng Anh cho trực quan:

| Số | Tiếng Việt | Tiếng Anh (định dạng LeetCode) |
|---:|---|---|
| 7 | bảy | `Seven` |
| 15 | mười lăm | `Fifteen` |
| 21 | hai mươi mốt | `Twenty One` |
| 40 | bốn mươi | `Forty` |
| 105 | một trăm linh (lẻ) năm | `One Hundred Five` |
| 1.000 | một nghìn (ngàn) | `One Thousand` |
| 12.345 | mười hai nghìn ba trăm bốn mươi lăm | `Twelve Thousand Three Hundred Forty Five` |
| 1.000.000 | một triệu | `One Million` |
| 1.000.010 | một triệu không trăm mười | `One Million Ten` |
| 1.234.567 | một triệu hai trăm ba mươi bốn nghìn năm trăm sáu mươi bảy | `One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven` |
| 1.000.000.000 | một tỷ | `One Billion` |

Đọc kỹ dòng `1.234.567`: tiếng Việt nói "**một triệu** | **hai trăm ba mươi bốn nghìn** | **năm trăm sáu mươi bảy**", tiếng Anh nói "**One Million** | **Two Hundred Thirty Four Thousand** | **Five Hundred Sixty Seven**". Hai câu có **cùng một khung**!

### 2.1. Điểm giống nhau quan trọng nhất: nhóm 3 chữ số

Cả người Việt lẫn người Anh/Mỹ đều tách số thành **nhóm 3 chữ số tính từ phải sang**:

- Việt Nam viết `1.234.567` (ngăn bằng dấu chấm), Anh/Mỹ viết `1,234,567` (ngăn bằng dấu phẩy).
- Mỗi nhóm có một **cấp** (scale):

| Cấp | Giá trị | Tiếng Việt | Tiếng Anh |
|---|---:|---|---|
| nhóm đơn vị | `1` | (không nói gì) | (không nói gì) |
| nhóm nghìn | `10^3 = 1,000` | nghìn / ngàn | `Thousand` |
| nhóm triệu | `10^6 = 1,000,000` | triệu | `Million` |
| nhóm tỷ | `10^9 = 1,000,000,000` | tỷ | `Billion` |

Quy tắc chung của cả hai ngôn ngữ: **đọc giá trị của nhóm (một số 0..999) rồi nói tên cấp**, đi từ nhóm lớn nhất xuống nhóm đơn vị.

Python có sẵn cách in số kèm dấu phẩy ngăn nhóm: trong f-string viết `f"{n:,}"` (dấu `,` sau dấu hai chấm nghĩa là "chèn dấu phẩy mỗi 3 chữ số"). Muốn kiểu Việt Nam thì thay `,` bằng `.` bằng `str.replace`.

In [2]:
for n in [123, 12345, 1234567, 1000010, 2147483647]:
    en = f"{n:,}"                 # kiểu Anh/Mỹ: dấu phẩy
    vi = en.replace(",", ".")     # kiểu Việt Nam: dấu chấm
    print(f"{n:>10} | EN: {en:>13} | VN: {vi:>13} | số nhóm = {en.count(',') + 1}")

       123 | EN:           123 | VN:           123 | số nhóm = 1
     12345 | EN:        12,345 | VN:        12.345 | số nhóm = 2
   1234567 | EN:     1,234,567 | VN:     1.234.567 | số nhóm = 3
   1000010 | EN:     1,000,010 | VN:     1.000.010 | số nhóm = 3
2147483647 | EN: 2,147,483,647 | VN: 2.147.483.647 | số nhóm = 4


### 2.2. Những điểm khác nhau — chính là những chỗ dễ sai khi code

| Hiện tượng | Tiếng Việt | Tiếng Anh | Hệ quả khi code |
|---|---|---|---|
| Số 10–19 | ghép đều đặn: mười, mười một, mười hai, mười lăm | **tên riêng**: `Ten, Eleven, Twelve, Thirteen, ..., Nineteen` | cần một bảng riêng `TEENS` |
| Hàng chục 20–90 | ghép đều đặn: hai mươi, ba mươi, bốn mươi | **tên riêng**: `Twenty, Thirty, Forty, Fifty, ...` | cần bảng `TENS` |
| Biến âm | mốt (21), lăm (15, 25), tư (24) | **không biến âm**: `Twenty One`, `Twenty Five` | tiếng Anh dễ hơn ở điểm này |
| Số 0 ở giữa nhóm | một trăm **linh** năm (105) | `One Hundred Five` — số 0 "im lặng" | chỉ đọc phần khác 0 |
| Nhóm toàn số 0 | 1.000.010: một triệu không trăm mười | `One Million Ten` — **bỏ hẳn** nhóm nghìn | không được in `"Thousand"` khi nhóm nghìn bằng 0 |
| Số 0 đứng một mình | không | `Zero` | trường hợp đặc biệt, xử lý riêng |

**Bẫy chính tả** (LeetCode so khớp từng chữ cái, sai 1 chữ là Wrong Answer):

- `Forty` — **không** có chữ `u` (dù `Four` có `u`). Viết `Fourty` là sai.
- `Fifty` (không phải `Fivety`), `Eighty` (một chữ `t`), `Ninety` (có chữ `e`).
- `Twelve`, `Thirteen`, `Fifteen`, `Eighteen` (một chữ `t`), `Nineteen`.

Bảng đầy đủ — học thuộc một lần là dùng mãi:

| | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| **ONES** (1–9) | — | One | Two | Three | Four | Five | Six | Seven | Eight | Nine |
| **TEENS** (10–19) | Ten | Eleven | Twelve | Thirteen | Fourteen | Fifteen | Sixteen | Seventeen | Eighteen | Nineteen |
| **TENS** (×10) | — | — | Twenty | Thirty | Forty | Fifty | Sixty | Seventy | Eighty | Ninety |

## 3. Kiến thức Python nền tảng

### 3.1. List làm "bảng tra" (lookup table)

Một **list** là dãy phần tử có thứ tự; phần tử thứ `i` (đếm từ 0) lấy bằng `lst[i]`, gọi là **indexing** (truy cập theo chỉ số). Nếu ta xếp các từ sao cho **chỉ số = con số cần đọc**, thì "đọc chữ số 7" chỉ là `ONES[7]` — không cần `if` nào cả. Đó là ý nghĩa của **bảng tra**.

Ba bảng ta dùng:

- `ONES[d]` với `d = 1..9`. Vị trí `0` để **chuỗi rỗng** `""` làm "chỗ giữ chỗ" (placeholder) — vì tiếng Anh không đọc chữ số 0 ở giữa số (nhớ: `105` → `One Hundred Five`, không có "Zero").
- `TEENS[n - 10]` với `n = 10..19`. Vì list đánh số từ 0, ta trừ 10: `TEENS[15 - 10] = TEENS[5] = "Fifteen"`.
- `TENS[t]` với `t = 2..9` là chữ số hàng chục: `TENS[4] = "Forty"`. Hai vị trí `0`, `1` để `""` vì số `0x` và `1x` không đọc bằng bảng này (`1x` thuộc `TEENS`).

Đặt chỗ giữ chỗ như vậy giúp **chỉ số khớp đúng con số**, đỡ phải cộng trừ lằng nhằng.

In [3]:
ONES = ["", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
TEENS = ["Ten", "Eleven", "Twelve", "Thirteen", "Fourteen",
         "Fifteen", "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
TENS = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]

print(len(ONES), len(TEENS), len(TENS))    # 10 10 10
print(ONES[7])           # Seven
print(TEENS[15 - 10])    # Fifteen
print(TENS[4])           # Forty   (không phải "Fourty"!)
print(repr(ONES[0]))     # ''      -> chỗ giữ chỗ, chuỗi rỗng

10 10 10
Seven
Fifteen
Forty
''


### 3.2. Toán tử `%` và `//` — tách chữ số và tách nhóm

- `a // b`: **chia lấy phần nguyên** (bỏ phần thập phân). Ví dụ `567 // 100 = 5`.
- `a % b`: **chia lấy dư** (modulo). Ví dụ `567 % 100 = 67`.

Với số tự nhiên, `// 10^k` là "**bỏ đi k chữ số cuối**", còn `% 10^k` là "**giữ lại k chữ số cuối**":

| Biểu thức | Kết quả | Ý nghĩa |
|---|---:|---|
| `67 // 10` | 6 | chữ số hàng chục |
| `67 % 10` | 7 | chữ số hàng đơn vị |
| `567 // 100` | 5 | chữ số hàng trăm |
| `567 % 100` | 67 | phần còn lại (< 100) |
| `1234567 % 1000` | 567 | **nhóm 3 chữ số cuối** (nhóm đơn vị) |
| `1234567 // 1000` | 1234 | bỏ nhóm cuối đi |
| `1234567 // 1000 % 1000` | 234 | nhóm nghìn |

Về thứ tự tính: `//` và `%` **cùng độ ưu tiên**, tính **từ trái sang phải**. Nên `1234567 // 1000 % 1000` nghĩa là `(1234567 // 1000) % 1000 = 1234 % 1000 = 234`.

Chú ý: `/` (một gạch) là phép chia **ra số thực**: `567 / 100 = 5.67` — không dùng để tách chữ số.

In [4]:
n = 67
print(n // 10, n % 10)          # 6 7   -> hàng chục, hàng đơn vị
n = 567
print(n // 100, n % 100)        # 5 67  -> hàng trăm, phần còn lại
print(n / 100)                  # 5.67  -> chia thực, KHÔNG dùng

big = 1234567
print(big % 1000)               # 567   -> nhóm đơn vị
print(big // 1000)              # 1234  -> bỏ nhóm cuối
print(big // 1000 % 1000)       # 234   -> nhóm nghìn
print(big // 1000**2)           # 1     -> nhóm triệu

6 7
5 67
5.67
567
1234
234
1


### 3.3. `divmod` — lấy thương và số dư trong một lần

`divmod(a, b)` trả về **cặp** `(a // b, a % b)`. Ta có thể "mở gói" (unpacking) cặp này vào hai biến cùng lúc:

```python
hundreds, rest = divmod(567, 100)   # hundreds = 5, rest = 67
```

Đọc là: "567 chia 100 được **5** (hàng trăm), dư **67** (phần còn lại)". Thứ tự luôn là **(thương, dư)** — nhớ: "chia trước, dư sau".

Mối liên hệ luôn đúng: `a == (a // b) * b + a % b`, ví dụ `567 == 5 * 100 + 67`. Nhờ đẳng thức này, khi ta đọc riêng `5` và `67` rồi ghép lại thì không mất thông tin gì.

In [5]:
print(divmod(567, 100))         # (5, 67)
print(divmod(1234567, 1000))    # (1234, 567)
print(divmod(40, 10))           # (4, 0)  -> chia hết, dư 0

hundreds, rest = divmod(567, 100)
print("hundreds =", hundreds, "| rest =", rest)
print("Kiểm tra lại:", hundreds * 100 + rest)   # 567

(5, 67)
(1234, 567)
(4, 0)
hundreds = 5 | rest = 67
Kiểm tra lại: 567


### 3.4. Vòng `while` tách số thành các nhóm 3 chữ số

Ghép `% 1000` và `// 1000` vào một vòng lặp, ta "bóc" dần từng nhóm từ **phải sang trái**:

```text
num = 1234567
  bóc: 1234567 % 1000 = 567,  num = 1234567 // 1000 = 1234
  bóc:    1234 % 1000 = 234,  num =    1234 // 1000 = 1
  bóc:       1 % 1000 = 1,    num =       1 // 1000 = 0   -> dừng
groups = [567, 234, 1]   (nhóm đơn vị đứng ĐẦU list)
```

Hai điều cần để ý:

1. List thu được xếp từ nhóm **thấp lên cao** (`groups[0]` là nhóm đơn vị, `groups[1]` là nhóm nghìn, ...). Điều này rất tiện: **chỉ số trong list = số thứ tự cấp**. Nhưng khi đọc phải đi từ cao xuống thấp, nên cuối cùng ta cần **đảo ngược**. Python có 3 cách: `lst.reverse()` (đảo tại chỗ, không trả về gì), `reversed(lst)` (trả về một "bộ duyệt" ngược) và `lst[::-1]` (tạo list mới đảo ngược).
2. Với `num = 0`, điều kiện `while num > 0` sai ngay từ đầu → list rỗng `[]`. Nghĩa là số 0 **không có nhóm nào để đọc** — ta sẽ phải xử lý riêng (`"Zero"`).

In [6]:
def split_groups(num: int) -> list:
    groups = []
    while num > 0:
        groups.append(num % 1000)   # giữ 3 chữ số cuối
        num //= 1000                # bỏ 3 chữ số cuối
    return groups


for n in [1234567, 1000010, 1000000, 999, 0]:
    print(f"{n:>8} -> {split_groups(n)}")

g = split_groups(1234567)
print()
print("reversed  :", list(reversed(g)))
print("slicing   :", g[::-1])
g.reverse()                 # đảo tại chỗ, trả về None
print("sau reverse():", g)

 1234567 -> [567, 234, 1]
 1000010 -> [10, 0, 1]
 1000000 -> [0, 0, 1]
     999 -> [999]
       0 -> []

reversed  : [1, 234, 567]
slicing   : [1, 234, 567]
sau reverse(): [1, 234, 567]


### 3.5. `str.join` và chuyện khoảng trắng thừa

`" ".join(list_of_strings)` dán các chuỗi trong list lại, **chèn đúng 1 dấu cách vào giữa** hai phần tử liền nhau:

```python
" ".join(["One", "Million", "Ten"])   # 'One Million Ten'
```

Nhưng `join` **không biết** phần tử nào là chuỗi rỗng — nó vẫn chèn dấu cách quanh chuỗi rỗng:

```python
" ".join(["One", "", "Thousand"])     # 'One  Thousand'  -> 2 dấu cách!
" ".join(["Twenty", ""])              # 'Twenty '        -> dấu cách ở cuối!
```

Đây là nguồn lỗi số 1 của bài này, vì các helper của ta sẽ trả về `""` khi gặp số 0. Có hai cách phòng:

1. **Lọc bỏ chuỗi rỗng trước khi join** bằng *list comprehension* (cách tạo list ngắn gọn): `[p for p in parts if p]` đọc là "lấy mọi `p` trong `parts` **nếu** `p` khác rỗng" (chuỗi rỗng được Python coi là `False`).
2. **Chỉ `append` khi có nội dung** — không bao giờ đưa `""` vào list.

Mẹo để **nhìn thấy** dấu cách: in bằng `repr(s)` (hiện cả dấu nháy) thay vì `print(s)`.

Một "băng dán cá nhân" hay gặp: `" ".join(s.split())` — `s.split()` (không truyền tham số) tách theo **mọi cụm khoảng trắng** và bỏ phần rỗng, nên ghép lại sẽ hết khoảng trắng thừa. Dùng được, nhưng tốt hơn là code đúng ngay từ đầu để hiểu rõ chỗ nào sinh ra chuỗi rỗng.

In [7]:
print(repr(" ".join(["One", "Million", "Ten"])))     # 'One Million Ten'
print(repr(" ".join(["One", "", "Thousand"])))       # 'One  Thousand'  (2 dấu cách)
print(repr(" ".join(["Twenty", ""])))                # 'Twenty '        (thừa ở cuối)

parts = ["One", "", "Thousand", ""]
clean = [p for p in parts if p]                      # lọc chuỗi rỗng
print(clean, "->", repr(" ".join(clean)))            # 'One Thousand'

messy = "  One   Hundred  "
print(repr(messy.split()), "->", repr(" ".join(messy.split())))

'One Million Ten'
'One  Thousand'
'Twenty '
['One', 'Thousand'] -> 'One Thousand'
['One', 'Hundred'] -> 'One Hundred'


## 4. Cách người mới hay nghĩ: đọc máy móc từng vị trí

Người mới thường nghĩ: "số có hàng trăm, hàng chục, hàng đơn vị — cứ đọc từng hàng là xong". Thử viết cho số `0 <= n < 1000`:

- Hàng trăm `h = n // 100` → `"<chữ số> Hundred"`.
- Hàng chục `t = n // 10 % 10` → tra bảng chục "ngây thơ" `["Zero", "Ten", "Twenty", ...]`.
- Hàng đơn vị `u = n % 10` → tra bảng `["Zero", "One", ..., "Nine"]`.

Code dưới đây làm đúng như vậy. Hãy xem nó đúng được bao nhiêu ca.

In [8]:
DIGIT_NAMES = ["Zero", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
TENS_NAIVE = ["Zero", "Ten", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]


def naive_by_position(n: int) -> str:
    # 0 <= n < 1000, đọc máy móc từng vị trí: trăm - chục - đơn vị
    h, t, u = n // 100, n // 10 % 10, n % 10
    return f"{DIGIT_NAMES[h]} Hundred {TENS_NAIVE[t]} {DIGIT_NAMES[u]}"


for n in [123, 115, 105, 100, 40]:
    print(f"{n:>3} -> {naive_by_position(n)}")

123 -> One Hundred Twenty Three
115 -> One Hundred Ten Five
105 -> One Hundred Zero Five
100 -> One Hundred Zero Zero
 40 -> Zero Hundred Forty Zero


### Cách ngây thơ vướng ở đâu?

| n | Cách ngây thơ in ra | Đúng phải là | Vướng mắc |
|---:|---|---|---|
| 123 | `One Hundred Twenty Three` | `One Hundred Twenty Three` | đúng (may mắn!) |
| 115 | `One Hundred Ten Five` | `One Hundred Fifteen` | **10–19 có tên riêng**, không ghép "Ten Five" |
| 105 | `One Hundred Zero Five` | `One Hundred Five` | **số 0 phải im lặng** |
| 100 | `One Hundred Zero Zero` | `One Hundred` | số 0 phải im lặng |
| 40 | `Zero Hundred Forty Zero` | `Forty` | không đọc **hàng trăm bằng 0** |

Và còn một vướng mắc lớn hơn: hàm này chỉ xử lý được `n < 1000`. Với số tới 2,1 tỷ, chẳng lẽ viết thêm hàng chục nhánh if cho hàng nghìn, chục nghìn, trăm nghìn, triệu...? Không cần — ở mục 1 ta đã thấy **mỗi nhóm 3 chữ số đọc giống hệt nhau**.

### Kế hoạch chia nhỏ (decomposition)

Mỗi tầng chỉ trả lời **một câu hỏi**, tầng trên dùng lại tầng dưới:

```text
Tầng 1: two_digits(n),   0 <= n < 100   "Đọc số có tối đa 2 chữ số?"   (xử lý teens, số 0 im lặng)
   │
   ▼
Tầng 2: three_digits(n), 0 <= n < 1000  "Đọc một nhóm 3 chữ số?"       (thêm "X Hundred" nếu có)
   │
   ▼
Tầng 3: ghép nhóm,       0 <= num < 2^31 "Đọc cả số?"                  (tách nhóm, gắn Thousand/Million/Billion,
                                                                       bỏ nhóm 0, xử lý num = 0)
```

Quy ước chung cho mọi helper: **đọc số 0 thì trả về chuỗi rỗng `""`** (vì số 0 bên trong một số lớn luôn "im lặng"). Chỉ có tầng ngoài cùng mới biến `num = 0` thành `"Zero"`.

## 5. Bước 1 — `two_digits(n)`: đọc số `0 <= n < 100`

Có 4 trường hợp **không chồng lên nhau** và **phủ kín** mọi `n` từ 0 đến 99:

| Trường hợp | Điều kiện | Cách đọc | Ví dụ |
|---|---|---|---|
| A | `n == 0` | `""` (im lặng) | `0` → `""` |
| B | `1 <= n <= 9` | `ONES[n]` | `7` → `Seven` |
| C | `10 <= n <= 19` | `TEENS[n - 10]` | `15` → `TEENS[5]` → `Fifteen` |
| D | `20 <= n <= 99` | `TENS[n // 10]`, rồi thêm `ONES[n % 10]` **nếu** `n % 10 != 0` | `42` → `Forty Two`; `40` → `Forty` |

Tính tay trường hợp D với `n = 42`: `42 // 10 = 4` → `TENS[4] = "Forty"`; `42 % 10 = 2` → `ONES[2] = "Two"` → `"Forty Two"`.
Với `n = 40`: `40 % 10 = 0` → **không** thêm gì → `"Forty"` (nếu vô tình thêm `ONES[0] = ""` bằng `+ " " +` sẽ thành `"Forty "` có dấu cách thừa).

Thứ tự kiểm tra `if n < 10` → `if n < 20` → còn lại: vì đã qua được `n < 10` thì chắc chắn `n >= 10`, nên `n < 20` nghĩa là đúng khoảng `10..19`. Viết các ngưỡng **tăng dần** giúp mỗi điều kiện ngắn gọn.

In [9]:
def two_digits(n: int) -> str:
    # Đọc 0 <= n < 100. Quy ước: n == 0 -> "" (im lặng).
    if n == 0:
        return ""
    if n < 10:                      # 1..9
        return ONES[n]
    if n < 20:                      # 10..19: tên riêng
        return TEENS[n - 10]
    tens_word = TENS[n // 10]       # 20..99
    ones_digit = n % 10
    if ones_digit == 0:
        return tens_word            # 20, 30, ..., 90
    return tens_word + " " + ONES[ones_digit]


for n in [0, 7, 10, 15, 19, 20, 40, 42, 99]:
    print(f"{n:>2} -> {two_digits(n)!r}")

 0 -> ''
 7 -> 'Seven'
10 -> 'Ten'
15 -> 'Fifteen'
19 -> 'Nineteen'
20 -> 'Twenty'
40 -> 'Forty'
42 -> 'Forty Two'
99 -> 'Ninety Nine'


## 6. Bước 2 — `three_digits(n)`: đọc một nhóm `0 <= n < 1000`

Dùng `divmod(n, 100)` tách thành **hàng trăm** và **phần còn lại**:

```python
hundreds, rest = divmod(n, 100)     # 0 <= hundreds <= 9, 0 <= rest <= 99
```

- Nếu `hundreds > 0` → có mảnh `ONES[hundreds] + " Hundred"`.
- Nếu `rest > 0` → có mảnh `two_digits(rest)` (tái sử dụng Bước 1!).
- Ghép các mảnh **đang có** bằng `" ".join(parts)`. Vì ta chỉ `append` khi có nội dung, list không bao giờ chứa `""` → không có dấu cách thừa.

Tính tay:

| n | hundreds | rest | Các mảnh | Kết quả |
|---:|---:|---:|---|---|
| 250 | 2 | 50 | `Two Hundred`, `Fifty` | `Two Hundred Fifty` |
| 105 | 1 | 5 | `One Hundred`, `Five` | `One Hundred Five` (không có "linh"!) |
| 100 | 1 | 0 | `One Hundred` | `One Hundred` |
| 42 | 0 | 42 | `Forty Two` | `Forty Two` (không có "Zero Hundred") |
| 0 | 0 | 0 | (không có) | `""` |

So với cách ngây thơ: ba lỗi `Ten Five`, `Zero Five`, `Zero Hundred` đều biến mất, vì (1) `two_digits` có nhánh `TEENS`, (2) số 0 trả về `""` và không được append, (3) hàng trăm bằng 0 thì bỏ qua.

In [10]:
def three_digits(n: int) -> str:
    # Đọc 0 <= n < 1000. Quy ước: n == 0 -> "".
    hundreds, rest = divmod(n, 100)
    parts = []
    if hundreds > 0:
        parts.append(ONES[hundreds] + " Hundred")
    if rest > 0:
        parts.append(two_digits(rest))
    return " ".join(parts)


for n in [250, 105, 100, 42, 0, 999]:
    print(f"{n:>3} -> {three_digits(n)!r}")

250 -> 'Two Hundred Fifty'
105 -> 'One Hundred Five'
100 -> 'One Hundred'
 42 -> 'Forty Two'
  0 -> ''
999 -> 'Nine Hundred Ninety Nine'


### Trace `three_digits` — nhìn từng mảnh được tạo ra

Bảng dự kiến (cột "(không)" nghĩa là mảnh đó **không được thêm** vào `parts`):

| n | hundreds | rest | mảnh Hundred | `two_digits(rest)` | kết quả |
|---:|---:|---:|---|---|---|
| 0 | 0 | 0 | (không) | (không) | `''` |
| 7 | 0 | 7 | (không) | `Seven` | `'Seven'` |
| 40 | 0 | 40 | (không) | `Forty` | `'Forty'` |
| 100 | 1 | 0 | `One Hundred` | (không) | `'One Hundred'` |
| 115 | 1 | 15 | `One Hundred` | `Fifteen` | `'One Hundred Fifteen'` |
| 999 | 9 | 99 | `Nine Hundred` | `Ninety Nine` | `'Nine Hundred Ninety Nine'` |

Hàm trace in đúng các cột đó. Cuối cell có thêm một **kiểm tra vét cạn nhỏ**: nhóm chỉ có 1000 giá trị (0..999), nên ta chạy hết để chắc rằng (a) không chuỗi nào có 2 dấu cách liền nhau hoặc dấu cách đầu/cuối, (b) 1000 số cho ra 1000 chuỗi **khác nhau** (không có hai số bị đọc trùng).

In [11]:
def trace_three_digits(n: int) -> str:
    hundreds, rest = divmod(n, 100)
    hundred_part = ONES[hundreds] + " Hundred" if hundreds > 0 else "(không)"
    rest_part = two_digits(rest) if rest > 0 else "(không)"
    result = three_digits(n)
    print(f"n={n:>3} | hundreds={hundreds} rest={rest:>2} | "
          f"{hundred_part:<12} + {rest_part:<11} -> {result!r}")
    return result


for n in [0, 7, 40, 100, 115, 999]:
    trace_three_digits(n)

all_groups = [three_digits(n) for n in range(1000)]
bad = [s for s in all_groups if "  " in s or s != s.strip()]
print()
print("Số chuỗi có khoảng trắng lỗi:", len(bad))
print("Số chuỗi khác nhau trên 1000 số:", len(set(all_groups)))

n=  0 | hundreds=0 rest= 0 | (không)      + (không)     -> ''
n=  7 | hundreds=0 rest= 7 | (không)      + Seven       -> 'Seven'
n= 40 | hundreds=0 rest=40 | (không)      + Forty       -> 'Forty'
n=100 | hundreds=1 rest= 0 | One Hundred  + (không)     -> 'One Hundred'
n=115 | hundreds=1 rest=15 | One Hundred  + Fifteen     -> 'One Hundred Fifteen'
n=999 | hundreds=9 rest=99 | Nine Hundred + Ninety Nine -> 'Nine Hundred Ninety Nine'

Số chuỗi có khoảng trắng lỗi: 0
Số chuỗi khác nhau trên 1000 số: 1000


## 7. Bước 3 — Ghép các nhóm: Cách 1 hoàn chỉnh (cách chính)

Giờ ta đã đọc được mọi nhóm 0..999. Phần còn lại:

1. **Trường hợp đặc biệt**: `num == 0` → trả về `"Zero"` ngay (vì vòng `while` sẽ không chạy lần nào).
2. Tạo bảng tên cấp theo **chỉ số nhóm**: `SCALES = ["", "Thousand", "Million", "Billion"]`. Nhóm số 0 (đơn vị) không có tên cấp nên để `""`.
3. Vòng `while num > 0`: bóc `chunk = num % 1000`, cắt `num //= 1000`.
   - Nếu `chunk > 0`: đọc `three_digits(chunk)`, gắn thêm tên cấp `SCALES[scale_index]` (nếu khác rỗng), rồi `append` vào `parts`.
   - Nếu `chunk == 0`: **bỏ qua** — không đọc, không gắn tên cấp. Đây chính là lý do `1,000,010` đọc là `One Million Ten` chứ không phải `One Million Thousand Ten`.
   - **Luôn** tăng `scale_index += 1`, kể cả khi bỏ qua nhóm (vì nhóm rỗng vẫn chiếm một cấp — nếu quên, các cấp phía trên sẽ bị "tụt" xuống).
4. `parts` đang xếp từ nhóm thấp lên cao → `reverse()` rồi `" ".join(parts)`.

Đây là phiên bản ta sẽ nộp, nên viết rõ ràng, mỗi dòng một việc.

In [12]:
SCALES = ["", "Thousand", "Million", "Billion"]


def number_to_words_groups(num: int) -> str:
    if num == 0:
        return "Zero"
    parts = []          # các cụm đã đọc, xếp từ nhóm THẤP lên CAO
    scale_index = 0     # 0: đơn vị, 1: Thousand, 2: Million, 3: Billion
    while num > 0:
        chunk = num % 1000
        num //= 1000
        if chunk > 0:                       # nhóm rỗng thì bỏ qua hẳn
            words = three_digits(chunk)
            if SCALES[scale_index]:
                words += " " + SCALES[scale_index]
            parts.append(words)
        scale_index += 1                    # luôn tăng, kể cả nhóm rỗng
    parts.reverse()                         # đọc từ nhóm cao xuống thấp
    return " ".join(parts)


for num, expected in examples:
    result = number_to_words_groups(num)
    print(f"{num:>8} -> {result!r}  {'OK' if result == expected else 'SAI'}")

     123 -> 'One Hundred Twenty Three'  OK
   12345 -> 'Twelve Thousand Three Hundred Forty Five'  OK
 1234567 -> 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'  OK


### Trace Cách 1 — tính tay trước

**Ví dụ 3: `num = 1234567`**

| Vòng | num trước | `chunk = num % 1000` | num sau (`// 1000`) | scale_index | tên cấp | Thêm vào `parts` |
|---:|---:|---:|---:|---:|---|---|
| 1 | 1234567 | 567 | 1234 | 0 | `''` | `'Five Hundred Sixty Seven'` |
| 2 | 1234 | 234 | 1 | 1 | `Thousand` | `'Two Hundred Thirty Four Thousand'` |
| 3 | 1 | 1 | 0 | 2 | `Million` | `'One Million'` |

`num` về 0 → dừng. Đảo `parts` rồi nối: `'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'`.

**Edge case: `num = 1000010`** (`1,000,010` — nhóm nghìn toàn số 0)

| Vòng | num trước | chunk | num sau | scale_index | tên cấp | Thêm vào `parts` |
|---:|---:|---:|---:|---:|---|---|
| 1 | 1000010 | 10 | 1000 | 0 | `''` | `'Ten'` |
| 2 | 1000 | 0 | 1 | 1 | `Thousand` | **bỏ qua** (nhóm rỗng) |
| 3 | 1 | 1 | 0 | 2 | `Million` | `'One Million'` |

Kết quả `'One Million Ten'`. Để ý vòng 2: tuy bỏ qua nhưng `scale_index` vẫn tăng lên 2 ở vòng 3, nên số `1` được gắn đúng `Million`.

In [13]:
def trace_number_to_words_groups(num: int) -> str:
    print(f"=== num = {num} ({num:,}) ===")
    if num == 0:
        print("num == 0 -> 'Zero'")
        return "Zero"
    parts, scale_index, step = [], 0, 1
    while num > 0:
        chunk = num % 1000
        new_num = num // 1000
        scale = SCALES[scale_index]
        if chunk > 0:
            words = three_digits(chunk) + (" " + scale if scale else "")
            parts.append(words)
            action = f"thêm {words!r}"
        else:
            action = "BỎ QUA nhóm rỗng"
        print(f"Vòng {step}: num={num:>7} chunk={chunk:>3} num mới={new_num:>4} "
              f"scale_index={scale_index} ({scale!r}) -> {action}")
        num, scale_index, step = new_num, scale_index + 1, step + 1
    print("parts (thấp -> cao):", parts)
    result = " ".join(reversed(parts))
    print("Kết quả:", repr(result))
    return result


_ = trace_number_to_words_groups(1234567)
print()
_ = trace_number_to_words_groups(1000010)

=== num = 1234567 (1,234,567) ===
Vòng 1: num=1234567 chunk=567 num mới=1234 scale_index=0 ('') -> thêm 'Five Hundred Sixty Seven'
Vòng 2: num=   1234 chunk=234 num mới=   1 scale_index=1 ('Thousand') -> thêm 'Two Hundred Thirty Four Thousand'
Vòng 3: num=      1 chunk=  1 num mới=   0 scale_index=2 ('Million') -> thêm 'One Million'
parts (thấp -> cao): ['Five Hundred Sixty Seven', 'Two Hundred Thirty Four Thousand', 'One Million']
Kết quả: 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'

=== num = 1000010 (1,000,010) ===
Vòng 1: num=1000010 chunk= 10 num mới=1000 scale_index=0 ('') -> thêm 'Ten'
Vòng 2: num=   1000 chunk=  0 num mới=   1 scale_index=1 ('Thousand') -> BỎ QUA nhóm rỗng
Vòng 3: num=      1 chunk=  1 num mới=   0 scale_index=2 ('Million') -> thêm 'One Million'
parts (thấp -> cao): ['Ten', 'One Million']
Kết quả: 'One Million Ten'


## 8. Code nộp LeetCode — Cách 1

Chuyển sang đúng format LeetCode:

- Mọi thứ nằm trong `class Solution`, method chính là `numberToWords(self, num: int) -> str`.
- `self` là tham số đầu tiên của mọi method, đại diện cho chính đối tượng `Solution`. Nhờ `self`, method này gọi được method khác (`self.three_digits(...)`) và đọc được các bảng (`self.ONES`).
- Ba bảng từ và `SCALES` đặt làm **thuộc tính của class** (class attribute): khai báo một lần ngay dưới dòng `class`, mọi method đều dùng chung.
- Thay `parts.reverse()` + `join` bằng `" ".join(reversed(parts))` — cùng kết quả, gọn hơn một dòng.

Code nộp **không in gì** (không `print`), chỉ `return`.

In [14]:
class Solution:
    ONES = ["", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
    TEENS = ["Ten", "Eleven", "Twelve", "Thirteen", "Fourteen",
             "Fifteen", "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
    TENS = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]
    SCALES = ["", "Thousand", "Million", "Billion"]

    def numberToWords(self, num: int) -> str:
        if num == 0:
            return "Zero"
        parts = []
        scale_index = 0
        while num > 0:
            chunk = num % 1000
            num //= 1000
            if chunk > 0:
                words = self.three_digits(chunk)
                if self.SCALES[scale_index]:
                    words += " " + self.SCALES[scale_index]
                parts.append(words)
            scale_index += 1
        return " ".join(reversed(parts))

    def two_digits(self, n: int) -> str:
        if n == 0:
            return ""
        if n < 10:
            return self.ONES[n]
        if n < 20:
            return self.TEENS[n - 10]
        if n % 10 == 0:
            return self.TENS[n // 10]
        return self.TENS[n // 10] + " " + self.ONES[n % 10]

    def three_digits(self, n: int) -> str:
        hundreds, rest = divmod(n, 100)
        parts = []
        if hundreds > 0:
            parts.append(self.ONES[hundreds] + " Hundred")
        if rest > 0:
            parts.append(self.two_digits(rest))
        return " ".join(parts)

In [15]:
edge_cases = [
    (0, "Zero"),
    (5, "Five"),
    (10, "Ten"),
    (13, "Thirteen"),
    (19, "Nineteen"),
    (20, "Twenty"),
    (21, "Twenty One"),
    (99, "Ninety Nine"),
    (100, "One Hundred"),
    (101, "One Hundred One"),
    (110, "One Hundred Ten"),
    (999, "Nine Hundred Ninety Nine"),
    (1000, "One Thousand"),
    (1001, "One Thousand One"),
    (1010, "One Thousand Ten"),
    (50868, "Fifty Thousand Eight Hundred Sixty Eight"),
    (100000, "One Hundred Thousand"),
    (1000000, "One Million"),
    (1000010, "One Million Ten"),
    (1001000, "One Million One Thousand"),
    (1000000000, "One Billion"),
    (1000000001, "One Billion One"),
    (2147483647, "Two Billion One Hundred Forty Seven Million Four Hundred "
                 "Eighty Three Thousand Six Hundred Forty Seven"),
]

tests = examples + edge_cases
sol = Solution()
for num, expected in tests:
    result = sol.numberToWords(num)
    assert result == expected, (num, result, expected)
    assert number_to_words_groups(num) == expected, num

print(f"Solution (Cách 1): PASS {len(tests)} test (3 ví dụ đề + {len(edge_cases)} edge case)")
print("Ví dụ cạnh biên:", repr(sol.numberToWords(2**31 - 1)))

Solution (Cách 1): PASS 26 test (3 ví dụ đề + 23 edge case)
Ví dụ cạnh biên: 'Two Billion One Hundred Forty Seven Million Four Hundred Eighty Three Thousand Six Hundred Forty Seven'


### Độ phức tạp của Cách 1

Gọi `d` là **số chữ số** của `num` (`d = ⌊log10(num)⌋ + 1`; theo constraints `d <= 10`).

- **Thời gian `O(d)`**: vòng `while` chạy `⌈d / 3⌉` lần (mỗi lần bóc 3 chữ số; tối đa 4 lần với `num < 2^31`). Mỗi lần gọi `three_digits` chỉ làm vài phép chia và tra bảng → `O(1)`. Bước `join` cuối cùng tỉ lệ với độ dài chuỗi kết quả, mà chuỗi này cũng chỉ `O(d)` từ.
- **Bộ nhớ `O(d)`**: list `parts` có tối đa 4 phần tử và chuỗi kết quả có `O(d)` từ. Các bảng từ có kích thước cố định (không phụ thuộc `num`).

Vì `d <= 10` theo đề, trong thực tế cả hai đều là **hằng số** — bài này không "khó" về tốc độ mà khó về **độ chính xác từng chi tiết**.

## 9. Kiến thức nền: Đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** với một bài toán **nhỏ hơn**. Mọi hàm đệ quy đúng đều có 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán đủ nhỏ để trả lời ngay, **không** gọi lại nữa. Thiếu base case → gọi vô tận → lỗi `RecursionError`.
2. **Recursive case** (trường hợp đệ quy): chia bài toán thành phần nhỏ hơn, gọi lại chính hàm đó cho phần nhỏ, rồi **kết hợp** kết quả.

Ví dụ mini: tổng các chữ số của `1234`.

- Base case: `n < 10` (chỉ 1 chữ số) → tổng chính là `n`.
- Recursive case: `digit_sum(n) = (n % 10) + digit_sum(n // 10)` — "chữ số cuối" cộng "tổng chữ số của phần còn lại". Ví dụ `digit_sum(1234) = 4 + digit_sum(123)`.

Khi hàm gọi lại chính nó, Python **tạm dừng** lời gọi hiện tại, cất vào **call stack** (ngăn xếp lời gọi), chạy lời gọi con cho xong rồi mới quay lại. Trong trace dưới, mỗi mức thụt lề là một tầng của call stack: dòng `=>` in ra lúc lời gọi con **trả kết quả về** cho tầng trên.

Liên hệ với bài của ta: đọc `1234567` = đọc `1` + `"Million"` + đọc `234567` — đây đúng là dạng "kết hợp kết quả của các bài nhỏ hơn"!

In [16]:
def digit_sum(n: int, depth: int = 0) -> int:
    pad = "    " * depth                    # thụt lề theo độ sâu call stack
    if n < 10:                              # base case
        print(f"{pad}digit_sum({n}) = {n}   <- base case, trả về ngay")
        return n
    print(f"{pad}digit_sum({n}) = {n % 10} + digit_sum({n // 10})")
    result = n % 10 + digit_sum(n // 10, depth + 1)   # recursive case
    print(f"{pad}=> digit_sum({n}) = {result}")
    return result


print("Kết quả cuối:", digit_sum(1234))

digit_sum(1234) = 4 + digit_sum(123)
    digit_sum(123) = 3 + digit_sum(12)
        digit_sum(12) = 2 + digit_sum(1)
            digit_sum(1) = 1   <- base case, trả về ngay
        => digit_sum(12) = 3
    => digit_sum(123) = 6
=> digit_sum(1234) = 10
Kết quả cuối: 10


## 10. Cách 2 — Đệ quy theo ngưỡng `20, 100, 1000, 10^6, 10^9`

Cách 1 cần 3 hàm và một vòng lặp. Để ý rằng **mọi tầng đều có cùng một khuôn**: "phần lớn + từ + phần nhỏ". Ta gom hết vào **một** hàm đệ quy `helper(n)`:

| Ngưỡng | Điều kiện | Công thức | Ví dụ |
|---|---|---|---|
| base | `n == 0` | `""` | `helper(0) = ""` |
| base | `n < 20` | `BELOW_20[n]` | `helper(15) = "Fifteen"` |
| 100 | `n < 100` | `TENS[n // 10]` + `helper(n % 10)` | `helper(42) = "Forty" + "Two"` |
| 1000 | `n < 1000` | `helper(n // 100)` + `"Hundred"` + `helper(n % 100)` | `567 = helper(5) Hundred helper(67)` |
| 10^6 | `n < 10**6` | `helper(n // 1000)` + `"Thousand"` + `helper(n % 1000)` | `234567 = helper(234) Thousand helper(567)` |
| 10^9 | `n < 10**9` | `helper(n // 10**6)` + `"Million"` + `helper(n % 10**6)` | |
| còn lại | `n >= 10**9` | `helper(n // 10**9)` + `"Billion"` + `helper(n % 10**9)` | `2147483647 = helper(2) Billion ...` |

Vì sao đệ quy **luôn dừng**? Ở mọi nhánh, các số truyền vào lời gọi con (`n // 1000`, `n % 1000`, ...) đều **nhỏ hơn hẳn** `n`, nên cứ đi xuống mãi sẽ chạm base case `n < 20`.

Hai chi tiết kỹ thuật:

- `BELOW_20 = ONES + TEENS`: phép `+` trên hai list là **nối list**, cho list 20 phần tử với `BELOW_20[n]` đúng là tên của `n` (0..19). Gộp lại giúp một nhánh `n < 20` thay cho hai nhánh `n < 10` / `n < 20`.
- `helper(n % 10)`, `helper(n % 1000)`, ... **có thể bằng `helper(0) = ""`** (ví dụ `n = 40`, `n = 120000`). Nếu ghép bằng `+ " " +` sẽ sinh dấu cách thừa. Ta dùng hàm nhỏ `join_nonempty(parts)` = `" ".join([p for p in parts if p])` — lọc chuỗi rỗng trước khi nối (mục 3.5).

In [17]:
BELOW_20 = ONES + TEENS          # nối list: 10 + 10 = 20 phần tử


def join_nonempty(parts: list) -> str:
    return " ".join([p for p in parts if p])


def helper(n: int) -> str:
    if n == 0:
        return ""
    if n < 20:
        return BELOW_20[n]
    if n < 100:
        return join_nonempty([TENS[n // 10], helper(n % 10)])
    if n < 1000:
        return join_nonempty([helper(n // 100), "Hundred", helper(n % 100)])
    if n < 10**6:
        return join_nonempty([helper(n // 1000), "Thousand", helper(n % 1000)])
    if n < 10**9:
        return join_nonempty([helper(n // 10**6), "Million", helper(n % 10**6)])
    return join_nonempty([helper(n // 10**9), "Billion", helper(n % 10**9)])


def number_to_words_recursive(num: int) -> str:
    return "Zero" if num == 0 else helper(num)


print(len(BELOW_20), BELOW_20[15])
for num in [0, 40, 120000, 1000010, 1234567]:
    print(f"{num:>8} -> {number_to_words_recursive(num)!r}")

20 Fifteen
       0 -> 'Zero'
      40 -> 'Forty'
  120000 -> 'One Hundred Twenty Thousand'
 1000010 -> 'One Million Ten'
 1234567 -> 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'


### Trace Cách 2 — cây lời gọi đệ quy cho `1234567`

Vẽ tay trước. Mỗi mức thụt lề là một lời gọi con; dòng `=>` là kết quả trả về sau khi đã ghép:

```text
helper(1234567)            [< 10^9]  = helper(1) Million helper(234567)
    helper(1) -> 'One'
    helper(234567)         [< 10^6]  = helper(234) Thousand helper(567)
        helper(234)        [< 1000]  = helper(2) Hundred helper(34)
            helper(2) -> 'Two'
            helper(34)     [< 100]   = Thirty + helper(4)
                helper(4) -> 'Four'
            => 'Thirty Four'
        => 'Two Hundred Thirty Four'
        helper(567)        [< 1000]  = helper(5) Hundred helper(67)
            helper(5) -> 'Five'
            helper(67)     [< 100]   = Sixty + helper(7)
                helper(7) -> 'Seven'
            => 'Sixty Seven'
        => 'Five Hundred Sixty Seven'
    => 'Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'
=> 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'
```

So với Cách 1: Cách 1 đi **từ phải sang trái** bằng vòng lặp (nhóm đơn vị trước), còn Cách 2 đi **từ trái sang phải** (bóc phần lớn nhất trước) nên **không cần đảo ngược** list.

In [18]:
def trace_helper(n: int, depth: int = 0) -> str:
    pad = "    " * depth
    if n == 0:
        print(f"{pad}helper(0) -> ''")
        return ""
    if n < 20:
        print(f"{pad}helper({n}) -> {BELOW_20[n]!r}")
        return BELOW_20[n]
    if n < 100:
        print(f"{pad}helper({n})  [< 100] = {TENS[n // 10]} + helper({n % 10})")
        result = join_nonempty([TENS[n // 10], trace_helper(n % 10, depth + 1)])
    else:
        # chọn nhánh như helper: (ngưỡng, đơn vị, từ, nhãn in ra); float("inf") = vô cùng
        for limit, unit, word, label in [(1000, 100, "Hundred", "1000"),
                                         (10**6, 1000, "Thousand", "10^6"),
                                         (10**9, 10**6, "Million", "10^9"),
                                         (float("inf"), 10**9, "Billion", "vô cùng")]:
            if n < limit:
                break
        print(f"{pad}helper({n})  [< {label}] = helper({n // unit}) {word} helper({n % unit})")
        left = trace_helper(n // unit, depth + 1)
        right = trace_helper(n % unit, depth + 1)
        result = join_nonempty([left, word, right])
    print(f"{pad}=> {result!r}")
    return result


_ = trace_helper(1234567)

helper(1234567)  [< 10^9] = helper(1) Million helper(234567)
    helper(1) -> 'One'
    helper(234567)  [< 10^6] = helper(234) Thousand helper(567)
        helper(234)  [< 1000] = helper(2) Hundred helper(34)
            helper(2) -> 'Two'
            helper(34)  [< 100] = Thirty + helper(4)
                helper(4) -> 'Four'
            => 'Thirty Four'
        => 'Two Hundred Thirty Four'
        helper(567)  [< 1000] = helper(5) Hundred helper(67)
            helper(5) -> 'Five'
            helper(67)  [< 100] = Sixty + helper(7)
                helper(7) -> 'Seven'
            => 'Sixty Seven'
        => 'Five Hundred Sixty Seven'
    => 'Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'
=> 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'


### Nhận xét và độ phức tạp của Cách 2

- **Đúng vì**: mỗi nhánh dùng đẳng thức `n = (n // T) * T + n % T` với `T` là đơn vị của nhánh (100, 1000, 10^6, 10^9), và cách đọc tiếng Anh đúng là "đọc phần `n // T`, nói tên đơn vị, đọc phần dư". Phần dư `n % T` luôn nhỏ hơn `T` nên được đọc bởi một nhánh thấp hơn — giống hệt cách nhóm 3 chữ số.
- **Thời gian `O(d)`**: mỗi tầng ngưỡng tách `n` thành 2 phần và tổng số lời gọi tỉ lệ với số chữ số (với `1234567` là 11 lời gọi). **Bộ nhớ `O(d)`** cho call stack (độ sâu tối đa khoảng 6 tầng) và chuỗi kết quả. Với `d <= 10`, cả hai là hằng số.
- **Ưu điểm**: rất gọn, không cần đảo list, mỗi ngưỡng là một dòng.
- **Nhược điểm**: người mới khó hình dung luồng chạy hơn vòng lặp; nếu ghép chuỗi bằng `+ " " +` mà quên lọc chuỗi rỗng sẽ dính lỗi dấu cách kép (xem mục Lỗi thường gặp).

Bản nộp LeetCode của Cách 2 dưới đây đặt tên `SolutionRecursive` để không đè lên `Solution` của Cách 1 trong notebook — **khi nộp, đổi tên thành `Solution`**. Ở đây `helper` được viết là **hàm lồng** (nested function) bên trong method: nó "nhìn thấy" được các biến `below_20`, `tens` của method bao ngoài.

In [19]:
class SolutionRecursive:        # khi nộp LeetCode: đổi tên thành Solution
    def numberToWords(self, num: int) -> str:
        below_20 = ["", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight",
                    "Nine", "Ten", "Eleven", "Twelve", "Thirteen", "Fourteen", "Fifteen",
                    "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
        tens = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]

        def join(parts):
            return " ".join([p for p in parts if p])

        def helper(n):
            if n < 20:
                return below_20[n]          # below_20[0] = "" lo luôn trường hợp n == 0
            if n < 100:
                return join([tens[n // 10], helper(n % 10)])
            if n < 1000:
                return join([helper(n // 100), "Hundred", helper(n % 100)])
            for value, word in [(10**9, "Billion"), (10**6, "Million"), (1000, "Thousand")]:
                if n >= value:
                    return join([helper(n // value), word, helper(n % value)])

        return "Zero" if num == 0 else helper(num)


sol2 = SolutionRecursive()
for num, expected in tests:
    assert sol2.numberToWords(num) == expected, num
    assert number_to_words_recursive(num) == expected, num
print(f"Cách 2 (đệ quy): PASS {len(tests)} test")

Cách 2 (đệ quy): PASS 26 test


## 11. Cách 3 — Bảng `(giá trị, từ)` duyệt từ lớn xuống (greedy kiểu số La Mã)

### Ôn lại: đổi số sang số La Mã bằng greedy

Nếu bạn đã học bài Roman to Integer, chắc đã gặp chiều ngược lại (LeetCode 12 — Integer to Roman). Cách làm chuẩn là **greedy** (tham lam): xếp bảng `(giá trị, ký hiệu)` **từ lớn đến nhỏ**; với mỗi dòng, **còn trừ được thì cứ trừ** và ghi ký hiệu ra.

Ví dụ `1994`: lấy `M` (1000) còn 994 → `CM` (900) còn 94 → `XC` (90) còn 4 → `IV` (4) còn 0 → `MCMXCIV`.

Gọi là "tham lam" vì mỗi bước luôn chọn **giá trị lớn nhất còn dùng được**, không cần nhìn trước. Thử nhanh:

In [20]:
ROMAN = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
         (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]


def int_to_roman(num: int) -> str:
    out = []
    for value, symbol in ROMAN:
        while num >= value:         # còn trừ được thì cứ trừ
            out.append(symbol)
            num -= value
    return "".join(out)


print(int_to_roman(1994))    # MCMXCIV
print(int_to_roman(3000))    # MMM  -> La Mã LẶP ký hiệu 3 lần

MCMXCIV
MMM


### Áp dụng ý tưởng greedy cho tiếng Anh

Khác biệt then chốt: số La Mã **lặp ký hiệu** (`3000` → `MMM`), còn tiếng Anh **đọc số lần** (`3000` → `Three Thousand`). Vì vậy bảng của ta có **hai loại dòng**:

| Loại dòng | Ví dụ | Khi `num >= value` thì làm gì |
|---|---|---|
| **Đơn vị đếm** (`value >= 100`) | `(10^9, Billion)`, `(10^6, Million)`, `(1000, Thousand)`, `(100, Hundred)` | `count = num // value`; **đọc `count`** (gọi đệ quy lại chính hàm này, vì `count < 1000`), thêm `word`; rồi `num %= value` |
| **Từ đứng riêng** (`value < 100`) | `(90, Ninety)`, ..., `(20, Twenty)`, `(19, Nineteen)`, ..., `(1, One)` | thêm `word`; rồi `num -= value` |

Với phần `< 100`, mỗi dòng được dùng **tối đa một lần**: sau khi trừ một giá trị hàng chục (`>= 20`), phần còn lại `< 10`; còn nếu `num < 20` thì một dòng `BELOW_20` trừ thẳng về 0. Nên ta dùng `if` thay cho `while` của bản La Mã.

**Xây bảng bằng list comprehension** thay vì gõ tay 31 dòng:

- `range(9, 1, -1)` đếm **lùi** `9, 8, ..., 2` (tham số thứ 3 là bước nhảy `-1`; dừng **trước** 1). Từ đó `(10 * i, TENS[i])` cho `(90, Ninety), ..., (20, Twenty)`.
- `range(19, 0, -1)` cho `19, 18, ..., 1` → `(i, BELOW_20[i])`.

In [21]:
VALUE_WORDS = [(10**9, "Billion"), (10**6, "Million"), (1000, "Thousand"), (100, "Hundred")]
VALUE_WORDS += [(10 * i, TENS[i]) for i in range(9, 1, -1)]      # 90 .. 20
VALUE_WORDS += [(i, BELOW_20[i]) for i in range(19, 0, -1)]       # 19 .. 1

print("Số dòng trong bảng:", len(VALUE_WORDS))
print("5 dòng đầu :", VALUE_WORDS[:5])
print("Chỉ số 11..13 (chỗ nối TENS -> BELOW_20):", VALUE_WORDS[11:14])
print("3 dòng cuối:", VALUE_WORDS[-3:])

Số dòng trong bảng: 31
5 dòng đầu : [(1000000000, 'Billion'), (1000000, 'Million'), (1000, 'Thousand'), (100, 'Hundred'), (90, 'Ninety')]
Chỉ số 11..13 (chỗ nối TENS -> BELOW_20): [(20, 'Twenty'), (19, 'Nineteen'), (18, 'Eighteen')]
3 dòng cuối: [(3, 'Three'), (2, 'Two'), (1, 'One')]


In [22]:
def number_to_words_table(num: int) -> str:
    if num == 0:
        return "Zero"
    parts = []
    for value, word in VALUE_WORDS:                   # từ lớn xuống nhỏ
        if num >= value:
            if value >= 100:                          # đơn vị đếm: đọc số lần
                count = num // value
                parts.append(number_to_words_table(count))
                parts.append(word)
                num %= value
            else:                                     # từ đứng riêng
                parts.append(word)
                num -= value
    return " ".join(parts)


for num in [0, 40, 115, 1000010, 1234567]:
    print(f"{num:>8} -> {number_to_words_table(num)!r}")

       0 -> 'Zero'
      40 -> 'Forty'
     115 -> 'One Hundred Fifteen'
 1000010 -> 'One Million Ten'
 1234567 -> 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'


### Trace Cách 3 trên `1234567`

Chỉ ghi các dòng của bảng mà `num >= value` (các dòng khác bị bỏ qua):

| Dòng bảng | num trước | Hành động | Thêm vào `parts` | num sau |
|---|---:|---|---|---:|
| `(1000000, Million)` | 1234567 | `count = 1`, đọc `1` | `One`, `Million` | 234567 |
| `(1000, Thousand)` | 234567 | `count = 234`, đọc đệ quy `234` | `Two Hundred Thirty Four`, `Thousand` | 567 |
| `(100, Hundred)` | 567 | `count = 5`, đọc `5` | `Five`, `Hundred` | 67 |
| `(60, Sixty)` | 67 | từ đứng riêng | `Sixty` | 7 |
| `(7, Seven)` | 7 | từ đứng riêng | `Seven` | 0 |

Có 5 dòng được dùng, **26 dòng** còn lại của bảng bị bỏ qua (gồm cả `(10^9, Billion)` ở đầu). Lời gọi đệ quy `number_to_words_table(234)` bên trong tự chạy một vòng greedy nhỏ: `(100, Hundred)` → `Two Hundred`, `(30, Thirty)`, `(4, Four)`.

In [23]:
def trace_table(num: int) -> str:
    print(f"=== num = {num} ===")
    parts, skipped = [], 0
    for value, word in VALUE_WORDS:
        if num < value:
            skipped += 1
            continue
        before = num
        if value >= 100:
            count = num // value
            count_words = number_to_words_table(count)
            parts += [count_words, word]
            num %= value
            print(f"({value}, {word}): num={before:>7} -> count={count}, đọc count = {count_words!r}"
                  f" -> thêm {count_words!r}, {word!r} | num còn {num}")
        else:
            parts.append(word)
            num -= value
            print(f"({value}, {word}): num={before:>7} -> từ đứng riêng, thêm {word!r} | num còn {num}")
    print("Số dòng bảng bị bỏ qua (num < value):", skipped)
    result = " ".join(parts)
    print("Kết quả:", repr(result))
    return result


_ = trace_table(1234567)

=== num = 1234567 ===
(1000000, Million): num=1234567 -> count=1, đọc count = 'One' -> thêm 'One', 'Million' | num còn 234567
(1000, Thousand): num= 234567 -> count=234, đọc count = 'Two Hundred Thirty Four' -> thêm 'Two Hundred Thirty Four', 'Thousand' | num còn 567
(100, Hundred): num=    567 -> count=5, đọc count = 'Five' -> thêm 'Five', 'Hundred' | num còn 67
(60, Sixty): num=     67 -> từ đứng riêng, thêm 'Sixty' | num còn 7
(7, Seven): num=      7 -> từ đứng riêng, thêm 'Seven' | num còn 0
Số dòng bảng bị bỏ qua (num < value): 26
Kết quả: 'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven'


### Nhận xét và độ phức tạp của Cách 3

- **Đúng vì**: các dòng "đơn vị đếm" xử lý đúng như Cách 2 (`num = count * value + num % value`, đọc `count` rồi nói tên đơn vị). Đến khi `num < 100`, greedy chọn dòng hàng chục lớn nhất `<= num` rồi một dòng `< 20` — đúng cấu trúc "Forty Two" hoặc "Fifteen".
- **Thời gian**: mỗi lời gọi duyệt hết 31 dòng bảng → `O(31)`; có thêm tối đa vài lời gọi đệ quy cho `count` (mỗi đơn vị đếm một lần). Tổng vẫn là `O(1)` với constraints của đề (nói chặt hơn: `O(số dòng bảng × số đơn vị đếm)`). Có thể thêm `if num == 0: break` để dừng sớm, nhưng không cần thiết.
- **Bộ nhớ `O(d)`** cho `parts` và kết quả.
- **Ưu điểm lớn nhất**: **dữ liệu tách khỏi logic**. Muốn hỗ trợ số tới `10^15`? Chỉ cần thêm dòng `(10**12, "Trillion")` vào đầu bảng — không sửa một dòng code nào (bài tập tự thử).
- **Nhược điểm**: phải duyệt cả bảng mỗi lần; hai loại dòng xử lý khác nhau nên cần hiểu rõ ranh giới `value >= 100`.

## 12. Cách 4 (chiều ngược) — `words_to_number`: đọc chữ thành số để kiểm chứng

Ta đã có 3 cách đi chiều **số → chữ**. Làm sao tự tin cả 3 đều đúng với **mọi** số tới 2,1 tỷ, khi không thể viết tay đáp án cho từng số? Một mẹo kiểm thử rất mạnh: viết hàm **chiều ngược** `words_to_number` (chữ → số), rồi kiểm tra **round-trip** (đi một vòng quay về):

```text
words_to_number(number_to_words(n)) == n     với mọi n
```

### Kiến thức nền: `dict` — tra cứu theo từ

`dict` (dictionary, từ điển) lưu các cặp **key → value**; `d[key]` trả về value, `key in d` kiểm tra key có tồn tại không (trung bình `O(1)`). Ta tạo `WORD_VALUE` ánh xạ `"Seven" → 7`, `"Forty" → 40`, ... từ chính các bảng đã có, dùng `enumerate(lst)` (cho cặp `(chỉ số, phần tử)`).

### Ý tưởng đọc ngược: hai biến `current` và `total`

Đọc từng từ từ trái sang phải:

- `current`: giá trị của **nhóm đang đọc dở** (luôn < 1000).
- `total`: tổng các nhóm **đã chốt** bằng một từ chỉ cấp.

| Loại từ | Hành động | Ví dụ |
|---|---|---|
| từ số (`One`..`Nineteen`, `Twenty`..`Ninety`) | `current += giá trị` | `Thirty` → `current += 30` |
| `Hundred` | `current *= 100` | `Two Hundred` → `2 * 100 = 200` |
| `Thousand` / `Million` / `Billion` | `total += current * cấp`; `current = 0` | chốt nhóm |
| hết chuỗi | trả về `total + current` | nhóm đơn vị chưa có từ chốt |

Tính tay `"One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven"`:

| Từ | Loại | current | total |
|---|---|---:|---:|
| `One` | số 1 | 1 | 0 |
| `Million` | cấp 10^6 | 0 | 1000000 |
| `Two` | số 2 | 2 | 1000000 |
| `Hundred` | ×100 | 200 | 1000000 |
| `Thirty` | số 30 | 230 | 1000000 |
| `Four` | số 4 | 234 | 1000000 |
| `Thousand` | cấp 10^3 | 0 | 1234000 |
| `Five` | số 5 | 5 | 1234000 |
| `Hundred` | ×100 | 500 | 1234000 |
| `Sixty` | số 60 | 560 | 1234000 |
| `Seven` | số 7 | 567 | 1234000 |

Hết chuỗi: `total + current = 1234000 + 567 = 1234567`.

In [24]:
WORD_VALUE = {}
for i, w in enumerate(BELOW_20):
    if w:                               # bỏ chỗ giữ chỗ ""
        WORD_VALUE[w] = i               # "One" -> 1, ..., "Nineteen" -> 19
for i, w in enumerate(TENS):
    if w:
        WORD_VALUE[w] = 10 * i          # "Twenty" -> 20, ..., "Ninety" -> 90
SCALE_VALUE = {"Thousand": 10**3, "Million": 10**6, "Billion": 10**9}


def words_to_number(s: str) -> int:
    if s == "Zero":
        return 0
    total = 0       # tổng các nhóm đã chốt
    current = 0     # nhóm đang đọc dở (< 1000)
    for w in s.split(" "):
        if w in WORD_VALUE:
            current += WORD_VALUE[w]
        elif w == "Hundred":
            current *= 100
        elif w in SCALE_VALUE:
            total += current * SCALE_VALUE[w]
            current = 0
        else:
            raise ValueError(f"Từ lạ: {w!r} trong {s!r}")
    return total + current


print(len(WORD_VALUE), WORD_VALUE["Seven"], WORD_VALUE["Forty"], WORD_VALUE["Nineteen"])
for num, text in examples:
    print(f"{text!r} -> {words_to_number(text)}")

27 7 40 19
'One Hundred Twenty Three' -> 123
'Twelve Thousand Three Hundred Forty Five' -> 12345
'One Million Two Hundred Thirty Four Thousand Five Hundred Sixty Seven' -> 1234567


In [25]:
def trace_words_to_number(s: str) -> int:
    total, current = 0, 0
    print(f"{'từ':<9} | {'loại':<12} | {'current':>7} | {'total':>8}")
    for w in s.split(" "):
        if w in WORD_VALUE:
            current += WORD_VALUE[w]
            kind = f"số {WORD_VALUE[w]}"
        elif w == "Hundred":
            current *= 100
            kind = "x100"
        else:
            total += current * SCALE_VALUE[w]
            current = 0
            kind = f"cấp {SCALE_VALUE[w]}"
        print(f"{w:<9} | {kind:<12} | {current:>7} | {total:>8}")
    print(f"Hết chuỗi: total + current = {total} + {current} = {total + current}")
    return total + current


_ = trace_words_to_number(examples[2][1])

từ        | loại         | current |    total
One       | số 1         |       1 |        0
Million   | cấp 1000000  |       0 |  1000000
Two       | số 2         |       2 |  1000000
Hundred   | x100         |     200 |  1000000
Thirty    | số 30        |     230 |  1000000
Four      | số 4         |     234 |  1000000
Thousand  | cấp 1000     |       0 |  1234000
Five      | số 5         |       5 |  1234000
Hundred   | x100         |     500 |  1234000
Sixty     | số 60        |     560 |  1234000
Seven     | số 7         |     567 |  1234000
Hết chuỗi: total + current = 1234000 + 567 = 1234567


## 13. Kiểm thử chéo: 3 cách + round-trip + kiểm tra định dạng

Ta kiểm 4 điều cho mỗi số `n`:

1. **Ba cách số → chữ cho cùng kết quả** (cùng với hai bản `class`). Nếu một cách có bug, khả năng cả ba cùng sai **giống hệt nhau** là rất thấp vì chúng được viết theo ba ý tưởng khác nhau.
2. **Round-trip**: `words_to_number(kết quả) == n`. Nếu ta quên một nhóm, đọc sai cấp, hay đặt `Hundred` sai chỗ, số đọc ngược sẽ khác `n`.
3. **Định dạng**: không có dấu cách đầu/cuối, không có 2 dấu cách liền nhau, mọi từ đều nằm trong **bộ từ vựng hợp lệ** `VOCAB`.
4. Kết hợp với các test viết tay ở mục 8 (đáp án độc lập).

Tập input:

- **Vét cạn** mọi `n` từ `0` đến `19 999` — phủ hết mọi giá trị 0..999 của **nhóm đơn vị**, mỗi giá trị đi kèm 20 giá trị nhóm nghìn khác nhau (0..19, gồm cả teens). Thêm vòng `g * 1000` và `g * 10^6 + g` với mọi `g` từ 0 đến 999 để mọi giá trị nhóm cũng xuất hiện ở **vị trí nghìn** và **vị trí triệu**.
- **Ngẫu nhiên có seed**: `random.seed(273)` để lần chạy nào cũng ra cùng một dãy số (tái lập được khi debug). Mẹo: nếu chọn đều trong `[0, 2^31 - 1]` thì gần như mọi số đều có 10 chữ số — ta sẽ không bao giờ thử số nhỏ. Vì vậy chọn **số chữ số `d` trước** (1..10), rồi mới chọn số có tối đa `d` chữ số.
- **Biên**: `0`, `2^31 - 1`, các lũy thừa của 1000, các số có nhóm 0 ở giữa.

**Giới hạn của round-trip** (cần biết để không tự tin thái quá): `words_to_number` dùng **chung bảng từ** với các hàm xuôi. Nếu bảng gõ nhầm `"Fourty"`, cả hai chiều cùng sai một cách nhất quán và round-trip **vẫn pass**! Vì vậy ta vẫn cần test viết tay với chuỗi đáp án chuẩn (mục 8) — xem minh họa ở phần Lỗi thường gặp.

In [26]:
import random

VOCAB = set(BELOW_20[1:]) | set(TENS[2:]) | {"Zero", "Hundred", "Thousand", "Million", "Billion"}
methods = [number_to_words_groups, number_to_words_recursive, number_to_words_table,
           Solution().numberToWords, SolutionRecursive().numberToWords]


def check_number(n: int) -> None:
    outs = [f(n) for f in methods]
    s = outs[0]
    assert all(o == s for o in outs), (n, outs)          # 1) các cách khớp nhau
    assert words_to_number(s) == n, (n, s)               # 2) round-trip
    assert s == s.strip() and "  " not in s, repr(s)     # 3) khoảng trắng
    assert all(w in VOCAB for w in s.split(" ")), s      # 3) từ vựng hợp lệ


for n in range(20000):                                   # vét cạn nhóm đơn vị
    check_number(n)
for g in range(1000):                                    # mọi nhóm ở vị trí nghìn, triệu
    check_number(g * 1000)
    check_number(g * 10**6 + g)

random.seed(273)
samples = []
for _ in range(3000):                                    # ngẫu nhiên có seed
    d = random.randint(1, 10)
    samples.append(random.randint(0, min(10**d - 1, 2**31 - 1)))
samples += [0, 2**31 - 1, 10**3, 10**6, 10**9, 1000010, 1001000, 1000000001, 2000000000]
for n in samples:
    check_number(n)

print("Vét cạn 0..19999 + mọi nhóm ở vị trí nghìn/triệu: PASS")
print(f"Ngẫu nhiên + biên ({len(samples)} số): PASS")
print("Vài mẫu ngẫu nhiên:", samples[:4])
print("Số lớn nhất đã thử:", max(samples))

Vét cạn 0..19999 + mọi nhóm ở vị trí nghìn/triệu: PASS
Ngẫu nhiên + biên (3009 số): PASS
Vài mẫu ngẫu nhiên: [9727, 4999, 1246658824, 224]
Số lớn nhất đã thử: 2147483647


## 14. Vì sao thuật toán đúng?

**1. Phân tích theo nhóm là duy nhất.** Mọi `num` (theo đề, `num < 2^31 < 10^12`) viết được **duy nhất** dưới dạng

```text
num = g3 * 10^9 + g2 * 10^6 + g1 * 10^3 + g0,   với 0 <= g0, g1, g2, g3 <= 999
```

Đây giống như viết số trong "hệ cơ số 1000": `g0 = num % 1000`, `g1 = (num // 1000) % 1000`, ... Vòng `while` của Cách 1 tính đúng lần lượt `g0, g1, g2, g3` — chính là mục 3.4.

**2. Quy tắc tiếng Anh.** Cách đọc `num` = lần lượt từ `g3` xuống `g0`, với mỗi nhóm **khác 0**: đọc nhóm (một số 1..999), rồi nói tên cấp. Nhóm bằng 0 **không đọc gì**. Riêng `num = 0` đọc là `Zero`.

**3. Invariant của vòng lặp Cách 1** (điều luôn đúng sau mỗi vòng): sau `k` vòng,
- `num` (biến đang chạy) `= num_ban_đầu // 1000^k` — phần **chưa xử lý** gồm các nhóm `g_k, g_{k+1}, ...`;
- `parts` chứa cách đọc (kèm tên cấp) của các nhóm **khác 0** trong `g_0 .. g_{k-1}`, theo thứ tự thấp → cao;
- `scale_index = k` — luôn khớp đúng cấp của nhóm sắp bóc (vì tăng **cả khi** nhóm bằng 0).

Khi vòng lặp dừng (`num = 0`), mọi nhóm đã xử lý; đảo `parts` cho đúng thứ tự cao → thấp → chính là quy tắc ở điểm 2.

**4. `three_digits` đúng với mọi nhóm.** `two_digits` chia `1..99` thành 3 khoảng rời nhau `[1, 9]`, `[10, 19]`, `[20, 99]` — phủ kín, không chồng chéo, mỗi khoảng đúng quy tắc tiếng Anh. `three_digits` thêm `"X Hundred"` khi hàng trăm khác 0. Vì miền chỉ có 1000 giá trị, ta còn **kiểm chứng vét cạn** được (mục 6 và mục 13) — với miền nhỏ, vét cạn là một "chứng minh" hoàn toàn hợp lệ.

**5. Không có khoảng trắng thừa**: mọi chỗ ghép đều dùng `" ".join` trên list **không chứa chuỗi rỗng** (chỉ `append` khi có nội dung, hoặc lọc bằng `join_nonempty`).

## 15. So sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Dễ hiểu | Khi nào dùng |
|---|---|---|---|---|---|
| Ngây thơ (đọc từng vị trí) | trăm - chục - đơn vị máy móc | `O(1)` | `O(1)` | rất dễ | **sai** với 10–19, số 0 — chỉ để thấy vướng mắc |
| **1. if/else + `three_digits` + ghép nhóm** | tầng < 100 → < 1000 → nhóm | `O(d)` | `O(d)` | dễ nhất | **nộp bài, phỏng vấn** — rõ ràng, dễ debug |
| 2. Đệ quy theo ngưỡng | `helper(n // T) + word + helper(n % T)` | `O(d)` | `O(d)` (call stack) | trung bình | khi muốn code ngắn |
| 3. Bảng `(giá trị, từ)` greedy | như số La Mã, đọc `count` thay vì lặp | `O(31 · số đơn vị)` = `O(1)` | `O(d)` | trung bình | khi muốn **mở rộng** (Trillion...) chỉ bằng dữ liệu |
| 4. Chữ → số (chiều ngược) | `current` / `total` | `O(số từ)` | `O(số từ)` | dễ | **kiểm chứng** round-trip, bài biến thể |

(`d` = số chữ số của `num`, `d <= 10` theo đề; vì thế mọi cách đều chạy tức thì — tiêu chí chọn là **độ rõ ràng và ít lỗi**.)

## 16. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `num = 0`

Vòng `while num > 0` không chạy lần nào → `parts` rỗng → trả về `""` thay vì `"Zero"`. Ngược lại, cũng **không được** cho các helper trả về `"Zero"` (xem Lỗi 5). **Sửa**: chỉ tầng ngoài cùng kiểm tra `if num == 0: return "Zero"`.

### Lỗi 2: In tên cấp cho nhóm rỗng

Gắn `SCALES[i]` mà không kiểm tra `chunk > 0` → `1000000` thành `"One Million  Thousand "` (thừa chữ `Thousand`, kèm dấu cách lung tung). **Sửa**: `if chunk > 0:` bao cả việc đọc lẫn việc gắn tên cấp.

### Lỗi 3: Chỉ tăng `scale_index` khi nhóm khác 0

Viết `scale_index += 1` **bên trong** `if chunk > 0` → khi gặp nhóm rỗng, cấp bị "tụt": `1000010` thành `"One Thousand Ten"` (sai hẳn giá trị!). **Sửa**: tăng `scale_index` ở **ngoài** `if`, mỗi vòng một lần.

### Lỗi 4: Dấu cách kép khi ghép đệ quy bằng `+ " " +`

Phiên bản đệ quy "kinh điển" hay thấy trên mạng ghép `helper(n // 1000) + " Thousand " + helper(n % 1000)` rồi `strip()` **một lần ở ngoài cùng**. `strip()` chỉ bỏ khoảng trắng ở **hai đầu**, còn dấu cách kép **ở giữa** thì vẫn còn: `120000` → `helper(120) = "One Hundred Twenty "` (thừa 1 dấu cách ở cuối vì `helper(0) = ""`), ghép tiếp `" Thousand "` → `"One Hundred Twenty  Thousand"`. **Sửa**: `strip()` ở **mọi** lần return, hoặc dùng `join_nonempty` như Cách 2.

Chạy cell dưới để thấy tận mắt cả 4 lỗi (dùng `repr` để nhìn rõ dấu cách):

In [27]:
def ntw_forget_zero(num):                   # Lỗi 1
    parts, i = [], 0
    while num > 0:
        num, chunk = divmod(num, 1000)
        if chunk > 0:
            parts.append(join_nonempty([three_digits(chunk), SCALES[i]]))
        i += 1
    return " ".join(reversed(parts))


def ntw_empty_group(num):                   # Lỗi 2: không kiểm tra chunk > 0
    parts, i = [], 0
    while num > 0:
        num, chunk = divmod(num, 1000)
        parts.append(three_digits(chunk) + (" " + SCALES[i] if SCALES[i] else ""))
        i += 1
    return " ".join(reversed(parts))


def ntw_scale_inside(num):                  # Lỗi 3: i += 1 nằm trong if
    parts, i = [], 0
    while num > 0:
        num, chunk = divmod(num, 1000)
        if chunk > 0:
            parts.append(join_nonempty([three_digits(chunk), SCALES[i]]))
            i += 1
    return " ".join(reversed(parts))


def helper_plus(n):                         # Lỗi 4: ghép bằng + " " +
    if n < 20:
        return BELOW_20[n]
    if n < 100:
        return TENS[n // 10] + " " + helper_plus(n % 10)
    if n < 1000:
        return helper_plus(n // 100) + " Hundred " + helper_plus(n % 100)
    return helper_plus(n // 1000) + " Thousand " + helper_plus(n % 1000)   # đủ cho n < 10^6


print("Lỗi 1:", repr(ntw_forget_zero(0)), "  (đúng: 'Zero')")
print("Lỗi 2:", repr(ntw_empty_group(1000000)), "  (đúng: 'One Million')")
print("Lỗi 3:", repr(ntw_scale_inside(1000010)), "  (đúng: 'One Million Ten')")
print("Lỗi 4:", repr(helper_plus(120000).strip()), "  (đúng: 'One Hundred Twenty Thousand')")

Lỗi 1: ''   (đúng: 'Zero')
Lỗi 2: 'One Million  Thousand '   (đúng: 'One Million')
Lỗi 3: 'One Thousand Ten'   (đúng: 'One Million Ten')
Lỗi 4: 'One Hundred Twenty  Thousand'   (đúng: 'One Hundred Twenty Thousand')


### Lỗi 5: Dùng `"Zero"` cho chữ số 0 bên trong số

Nếu bảng `ONES[0] = "Zero"` (giống bảng `DIGIT_NAMES` của cách ngây thơ) và quên kiểm tra `n % 10 == 0`, ta được `"Twenty Zero"`, `"One Hundred Zero"`. **Sửa**: chữ số 0 bên trong luôn im lặng — helper trả `""` cho 0; chỉ tầng ngoài biến `num = 0` thành `"Zero"`.

### Lỗi 6: Sai chính tả — và vì sao round-trip **không** bắt được

`"Fourty"` thay cho `"Forty"`, `"Ninty"` thay cho `"Ninety"`, `"Eightteen"`... LeetCode so từng ký tự nên sẽ báo Wrong Answer. Nguy hiểm ở chỗ: nếu hàm ngược dùng **chung bảng gõ sai**, round-trip vẫn pass (cell dưới minh họa). **Sửa**: luôn có vài test với **chuỗi đáp án viết tay** (như Ví dụ 2 có chữ `Forty`), và soát chính tả bảng bằng mắt một lần.

### Lỗi 7: Thêm thứ đề không yêu cầu

- `"One Hundred and Five"` (kiểu Anh-Anh), `"Twenty-Three"` (gạch nối), `"one hundred"` (chữ thường) → đều Wrong Answer. Đề chỉ chấp nhận định dạng như ví dụ.
- Dùng `str(num)` rồi xử lý từng ký tự cũng được, nhưng dễ rối chỉ số; với số, `%` / `//` / `divmod` tự nhiên hơn.

In [28]:
# Lỗi 5: bảng có "Zero" ở vị trí 0 và ghép mù quáng
ONES_WITH_ZERO = ["Zero"] + ONES[1:]
def two_digits_zero_bug(n):
    if n < 10:
        return ONES_WITH_ZERO[n]
    if n < 20:
        return TEENS[n - 10]
    return TENS[n // 10] + " " + ONES_WITH_ZERO[n % 10]
print("Lỗi 5:", repr(two_digits_zero_bug(20)), "  (đúng: 'Twenty')")

# Lỗi 6: bảng gõ sai "Fourty" -> round-trip vẫn pass, test viết tay mới bắt được
TENS_TYPO = TENS[:4] + ["Fourty"] + TENS[5:]
def two_digits_typo(n):
    return join_nonempty([TENS_TYPO[n // 10], ONES[n % 10]]) if n >= 20 else BELOW_20[n]
WORD_VALUE_TYPO = {}                       # dict chiều ngược, xây như mục 12
for i, w in enumerate(BELOW_20):
    if w:
        WORD_VALUE_TYPO[w] = i
for i, w in enumerate(TENS_TYPO):
    if w:
        WORD_VALUE_TYPO[w] = 10 * i

s = two_digits_typo(45)
print("Lỗi 6:", repr(s))
back = sum([WORD_VALUE_TYPO[w] for w in s.split()])      # đọc ngược (không có Hundred/cấp)
print("  round-trip với bảng gõ sai:", back == 45, "-> không phát hiện!")
print("  so với đáp án viết tay 'Forty Five':", s == "Forty Five", "-> phát hiện!")

Lỗi 5: 'Twenty Zero'   (đúng: 'Twenty')
Lỗi 6: 'Fourty Five'
  round-trip với bảng gõ sai: True -> không phát hiện!
  so với đáp án viết tay 'Forty Five': False -> phát hiện!


## 17. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận ra dạng bài này

- Đề yêu cầu **chuyển đổi định dạng** giữa số và chữ / ký hiệu (số ↔ chữ, số ↔ La Mã, số ↔ tên cột Excel): nghĩ ngay tới **bảng tra** (list theo chỉ số, hoặc list các cặp `(giá trị, ký hiệu)`) + tách số bằng `%` / `//`.
- Cấu trúc **lặp lại theo cấp** (cứ 3 chữ số lại một cấp Thousand/Million/Billion): viết **helper cho một khối** rồi lặp/đệ quy — đừng viết if cho từng cấp.
- Bài **mô phỏng nhiều edge case**: trước khi code, **liệt kê edge case ra giấy** (0, 10–19, bội của 10, bội của 100, nhóm rỗng ở giữa, giá trị lớn nhất). Sau khi code, test bằng **hàm ngược** (round-trip) + vét cạn miền nhỏ + ngẫu nhiên có seed.
- Ghép chuỗi có phần có thể rỗng: **gom vào list rồi `" ".join`**, không ghép bằng `+ " " +`.

### Bài liên quan trên LeetCode

| Bài | Độ khó | Liên hệ |
|---|---|---|
| 12. Integer to Roman | Medium | greedy bảng `(giá trị, ký hiệu)` — chính là Cách 3 |
| 13. Roman to Integer | Easy | chiều ngược (ký hiệu → số), giống Cách 4 |
| 1556. Thousand Separator | Easy | tách nhóm 3 chữ số, chèn dấu `.` (Bài tập 5) |
| 168. Excel Sheet Column Title | Easy | tách chữ số theo cơ số 26 bằng `%` / `//` |
| 171. Excel Sheet Column Number | Easy | chiều ngược của bài 168 |
| 68. Text Justification | Hard | bài mô phỏng nhiều chi tiết, khoảng trắng phải chính xác |
| 65. Valid Number | Hard | mô phỏng + liệt kê edge case kỹ lưỡng |

### Biến thể hay gặp khi phỏng vấn

- Đọc số **âm** (`"Negative ..."`), số **lớn hơn** (thêm `Trillion` — Cách 3 chỉ cần thêm một dòng bảng).
- Đọc **số tiền** có phần thập phân (`"... Dollars and Five Cents"`).
- Đọc số bằng **ngôn ngữ khác** (tiếng Việt — Bài tập 6), nơi quy tắc nhóm vẫn giữ nguyên nhưng cách đọc trong nhóm khác đi.

## 18. Bài tập vận dụng

Làm theo thứ tự, từ dễ đến khó. **Đừng xem đáp án** trước khi tự thử — đáp án nằm ở cuối notebook (mục 20), gợi ý theo mức ở mục 19.

### Bài 1 — Đoán output (không chạy code)

Đoán xem mỗi dòng in ra gì, rồi mới xem đáp án:

```python
print(1234567 % 1000, 1234567 // 1000)
print(divmod(50868, 1000))
print(TENS[4], TEENS[4], ONES[4])
print(repr(" ".join(["Fifty", "", "Thousand"])))
print(repr(" ".join([w for w in ["Fifty", "", "Thousand"] if w])))
print(split_groups(1000010))
```

### Bài 2 — Tính bằng tay

Với mỗi số dưới đây, hãy (a) viết các nhóm theo thứ tự `split_groups` trả về, (b) đọc từng nhóm khác 0 kèm tên cấp, (c) ghép ra chuỗi cuối cùng.

1. `50868`
2. `1001001`
3. `100000`
4. `2147483647`

Gợi ý trình bày: dùng bảng `nhóm | giá trị | đọc nhóm | tên cấp` giống mục 7.

### Bài 3 — Điền TODO: `three_digits_exercise`

Viết lại hàm đọc một nhóm `0 <= n <= 999` **trong một hàm duy nhất** (không gọi `two_digits`), dùng các bảng `ONES`, `TEENS`, `TENS`. Quy ước: `n == 0` trả về `""`.

In [29]:
def three_digits_exercise(n: int) -> str:
    parts = []
    hundreds, rest = divmod(n, 100)
    # TODO 1: nếu hundreds > 0, thêm ONES[hundreds] + " Hundred" vào parts
    # TODO 2: nếu 1 <= rest <= 9: thêm ONES[rest]
    # TODO 3: nếu 10 <= rest <= 19: thêm TEENS[rest - 10]
    # TODO 4: nếu rest >= 20: thêm TENS[rest // 10];
    #         và nếu rest % 10 != 0 thì thêm ONES[rest % 10]
    # TODO 5: return " ".join(parts)
    pass


# Tự làm xong rồi mới mở test:
# assert three_digits_exercise(0) == ""
# assert three_digits_exercise(7) == "Seven"
# assert three_digits_exercise(15) == "Fifteen"
# assert three_digits_exercise(40) == "Forty"
# assert three_digits_exercise(105) == "One Hundred Five"
# assert three_digits_exercise(999) == "Nine Hundred Ninety Nine"
# assert all(three_digits_exercise(n) == three_digits(n) for n in range(1000))

### Bài 4 — Tự viết lại toàn bộ lời giải

Không nhìn code phía trên, tự viết `number_to_words_exercise(num)` cho `0 <= num <= 2^31 - 1`. Được phép dùng `three_digits` (hoặc hàm của Bài 3). Nhớ 3 chi tiết: `Zero`, bỏ nhóm rỗng, `scale_index` luôn tăng.

In [30]:
def number_to_words_exercise(num: int) -> str:
    # TODO 1: num == 0 -> "Zero"
    # TODO 2: tạo SCALES = ["", "Thousand", "Million", "Billion"], parts = [], scale_index = 0
    # TODO 3: while num > 0: bóc chunk = num % 1000, num //= 1000
    # TODO 4:     nếu chunk > 0: đọc three_digits(chunk) + tên cấp (nếu có), append vào parts
    # TODO 5:     scale_index += 1  (ở đâu?)
    # TODO 6: đảo parts và " ".join
    pass


# Gợi ý test:
# for num, expected in tests:
#     assert number_to_words_exercise(num) == expected, (num, number_to_words_exercise(num))

### Bài 5 — Biến thể: LeetCode 1556. Thousand Separator (Easy)

Đề: cho số nguyên `n` (`0 <= n <= 2^31 - 1`), hãy thêm dấu chấm `"."` làm **dấu ngăn cách hàng nghìn** và trả về dạng chuỗi.

```text
Input: n = 987          Output: "987"
Input: n = 1234         Output: "1.234"
```

Signature trên LeetCode: `def thousandSeparator(self, n: int) -> str`. Ở đây viết dạng hàm thường.

Yêu cầu: **không** dùng `f"{n:,}"` — hãy tự tách nhóm bằng `% 1000` và `// 1000`. Chú ý: mọi nhóm **trừ nhóm đầu tiên** phải đủ 3 chữ số (`1000010` → `"1.000.010"`, không phải `"1.0.10"`). Gợi ý cú pháp: `f"{7:03d}"` cho `"007"` (độ rộng 3, đệm số 0 bên trái).

In [31]:
def thousand_separator(n: int) -> str:
    # TODO 1: nếu n < 1000 thì chỉ cần str(n)
    # TODO 2: bóc các nhóm % 1000; nhóm KHÔNG phải nhóm đầu thì định dạng f"{g:03d}"
    # TODO 3: nhóm đầu (cao nhất) viết bình thường bằng str
    # TODO 4: đảo thứ tự và nối bằng "."
    pass


# Gợi ý test:
# assert thousand_separator(987) == "987"
# assert thousand_separator(1234) == "1.234"
# assert thousand_separator(123456789) == "123.456.789"
# assert thousand_separator(0) == "0"
# assert thousand_separator(1000010) == "1.000.010"

### Bài 6 — Mini challenge: đọc số bằng **tiếng Việt**

Viết `vietnamese_words(num)` cho `0 <= num <= 2^31 - 1`, trả về chữ thường, các từ cách nhau 1 dấu cách. Khung nhóm giữ nguyên như tiếng Anh (cấp: `""`, `nghìn`, `triệu`, `tỷ`; bỏ nhóm toàn 0), nhưng cách đọc **trong** nhóm theo quy ước sau (quy ước của bài tập này, theo cách đọc phổ thông miền Bắc):

1. `0` → `"không"`.
2. **Nhóm đầu tiên** (cao nhất) đọc như số bình thường: `5` → `"năm"`, `15` → `"mười lăm"`.
3. **Các nhóm phía sau** (khác 0) đọc **đủ 3 vị trí**, kể cả hàng trăm bằng 0: `1005` → `"một nghìn không trăm linh năm"`.
4. Hàng chục bằng 0 nhưng hàng đơn vị khác 0, và **đã có** phần "trăm" phía trước → thêm `"linh"`: `105` → `"một trăm linh năm"`.
5. Hàng chục = 1 → `"mười"` (không phải "một mươi"); hàng chục >= 2 → `"<chữ số> mươi"`.
6. Hàng đơn vị: `1` đọc `"mốt"` khi hàng chục >= 2 (`21` → `"hai mươi mốt"`), còn lại `"một"` (`11` → `"mười một"`, `101` → `"một trăm linh một"`); `5` đọc `"lăm"` khi hàng chục >= 1 (`15`, `25`), còn lại `"năm"`; `4` luôn đọc `"bốn"`.

Ví dụ: `1000010` → `"một triệu không trăm mười"`; `1234567` → `"một triệu hai trăm ba mươi bốn nghìn năm trăm sáu mươi bảy"`.

In [32]:
VI_DIGITS = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
VI_SCALES = ["", "nghìn", "triệu", "tỷ"]


def vietnamese_words(num: int) -> str:
    # TODO 1: num == 0 -> "không"
    # TODO 2: viết helper đọc một nhóm g (0 < g < 1000) với cờ full
    #         (full = True nếu KHÔNG phải nhóm đầu tiên -> luôn đọc "x trăm")
    # TODO 3: tách nhóm như split_groups; duyệt từ nhóm cao xuống thấp, bỏ nhóm 0
    # TODO 4: gắn VI_SCALES[i], nối bằng " "
    pass


# Gợi ý test:
# assert vietnamese_words(0) == "không"
# assert vietnamese_words(15) == "mười lăm"
# assert vietnamese_words(21) == "hai mươi mốt"
# assert vietnamese_words(105) == "một trăm linh năm"
# assert vietnamese_words(1005) == "một nghìn không trăm linh năm"
# assert vietnamese_words(1000010) == "một triệu không trăm mười"
# assert vietnamese_words(1234567) == "một triệu hai trăm ba mươi bốn nghìn năm trăm sáu mươi bảy"

## 19. Gợi ý theo từng mức

Đọc gợi ý mức 1 trước; chỉ xem mức tiếp theo khi thật sự bí.

**Bài 1**
- Mức 1: `%` giữ chữ số cuối, `//` bỏ chữ số cuối; `divmod` trả về (thương, dư).
- Mức 2: `TEENS[4]` là phần tử thứ 5 của bảng teens, tức số `14`. `join` vẫn chèn dấu cách quanh chuỗi rỗng.
- Mức 3: `split_groups` trả nhóm từ thấp lên cao, nhóm rỗng vẫn có mặt (là số `0`).

**Bài 2**
- Mức 1: viết số có dấu phẩy ngăn nhóm trước (`50,868`, `1,001,001`, ...).
- Mức 2: nhóm nào bằng 0 thì bỏ qua cả cách đọc lẫn tên cấp.
- Mức 3: `2,147,483,647` có 4 nhóm: `2` Billion, `147` Million, `483` Thousand, `647`.

**Bài 3**
- Mức 1: đây chính là `two_digits` "nhét" vào trong `three_digits`.
- Mức 2: dùng chuỗi `if / elif / elif` theo `rest`: `1 <= rest <= 9`, `10 <= rest <= 19`, `rest >= 20`.
- Mức 3: nhánh cuối: `parts.append(TENS[rest // 10])` rồi `if rest % 10: parts.append(ONES[rest % 10])`.

**Bài 4**
- Mức 1: chép lại khung 3 tầng ở mục 4, chỉ viết tầng 3.
- Mức 2: `scale_index += 1` phải nằm **ngoài** `if chunk > 0`.
- Mức 3: `words = three_digits(chunk)`; `if SCALES[scale_index]: words += " " + SCALES[scale_index]`; cuối cùng `" ".join(reversed(parts))`.

**Bài 5**
- Mức 1: giống `split_groups`, nhưng thay vì đọc nhóm bằng chữ thì đổi nhóm thành chuỗi số.
- Mức 2: vòng `while n >= 1000` bóc các nhóm "phía sau" (đều cần 3 chữ số); khi thoát vòng, `n` còn lại chính là nhóm đầu.
- Mức 3: `groups.append(f"{n % 1000:03d}")`, sau vòng lặp `groups.append(str(n))`, rồi `".".join(reversed(groups))`.

**Bài 6**
- Mức 1: tách hai việc: (a) khung nhóm — copy từ Cách 1, đổi bảng cấp; (b) helper `vi_group(g, full)` đọc một nhóm.
- Mức 2: trong helper, `h, rest = divmod(g, 100)` và `t, u = divmod(rest, 10)`; nếu `full or h > 0` thì thêm `VI_DIGITS[h], "trăm"`. Nhóm đầu tiên là nhóm có chỉ số lớn nhất (`len(groups) - 1`).
- Mức 3: xét 3 trường hợp của `t`: `t == 0` (có thể thêm `"linh"` nếu list từ đã có phần trăm), `t == 1` (`"mười"` + `"lăm"`/chữ số), `t >= 2` (`"<t> mươi"` + `"mốt"`/`"lăm"`/chữ số).

## 20. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1 — chạy để kiểm tra dự đoán

Kết quả mong đợi: `567 1234`; `(50, 868)`; `Forty Fourteen Four`; `'Fifty  Thousand'` (2 dấu cách vì có chuỗi rỗng ở giữa); `'Fifty Thousand'`; `[10, 0, 1]` (nhóm nghìn là `0` vẫn có mặt trong list).

In [33]:
print(1234567 % 1000, 1234567 // 1000)
print(divmod(50868, 1000))
print(TENS[4], TEENS[4], ONES[4])
print(repr(" ".join(["Fifty", "", "Thousand"])))
print(repr(" ".join([w for w in ["Fifty", "", "Thousand"] if w])))
print(split_groups(1000010))

567 1234
(50, 868)
Forty Fourteen Four
'Fifty  Thousand'
'Fifty Thousand'
[10, 0, 1]


### Đáp án Bài 2 — tính tay

| Số | `split_groups` | Đọc các nhóm khác 0 (từ cao xuống) | Kết quả |
|---|---|---|---|
| `50,868` | `[868, 50]` | `Fifty Thousand` · `Eight Hundred Sixty Eight` | `Fifty Thousand Eight Hundred Sixty Eight` |
| `1,001,001` | `[1, 1, 1]` | `One Million` · `One Thousand` · `One` | `One Million One Thousand One` |
| `100,000` | `[0, 100]` | `One Hundred Thousand` · (nhóm đơn vị rỗng) | `One Hundred Thousand` |
| `2,147,483,647` | `[647, 483, 147, 2]` | `Two Billion` · `One Hundred Forty Seven Million` · `Four Hundred Eighty Three Thousand` · `Six Hundred Forty Seven` | `Two Billion One Hundred Forty Seven Million Four Hundred Eighty Three Thousand Six Hundred Forty Seven` |

Chú ý `100000`: nhóm nghìn là `100` → `One Hundred` + `Thousand`. Đừng nhầm "Hundred" là một cấp — nó nằm **bên trong** nhóm.

In [34]:
for n in [50868, 1001001, 100000, 2147483647]:
    print(f"{n:>13,} | groups = {split_groups(n)}")
    print(f"{'':>13} | {number_to_words_groups(n)}")

       50,868 | groups = [868, 50]
              | Fifty Thousand Eight Hundred Sixty Eight
    1,001,001 | groups = [1, 1, 1]
              | One Million One Thousand One
      100,000 | groups = [0, 100]
              | One Hundred Thousand
2,147,483,647 | groups = [647, 483, 147, 2]
              | Two Billion One Hundred Forty Seven Million Four Hundred Eighty Three Thousand Six Hundred Forty Seven


In [35]:
# ===== Đáp án Bài 3 =====
def answer_three_digits_exercise(n: int) -> str:
    parts = []
    hundreds, rest = divmod(n, 100)
    if hundreds > 0:
        parts.append(ONES[hundreds] + " Hundred")
    if 1 <= rest <= 9:
        parts.append(ONES[rest])
    elif 10 <= rest <= 19:
        parts.append(TEENS[rest - 10])
    elif rest >= 20:
        parts.append(TENS[rest // 10])
        if rest % 10 != 0:
            parts.append(ONES[rest % 10])
    return " ".join(parts)


assert answer_three_digits_exercise(0) == ""
assert answer_three_digits_exercise(7) == "Seven"
assert answer_three_digits_exercise(15) == "Fifteen"
assert answer_three_digits_exercise(40) == "Forty"
assert answer_three_digits_exercise(105) == "One Hundred Five"
assert answer_three_digits_exercise(999) == "Nine Hundred Ninety Nine"
assert all(answer_three_digits_exercise(n) == three_digits(n) for n in range(1000))


# ===== Đáp án Bài 4 =====
def answer_number_to_words_exercise(num: int) -> str:
    if num == 0:
        return "Zero"
    scales = ["", "Thousand", "Million", "Billion"]
    parts, scale_index = [], 0
    while num > 0:
        chunk = num % 1000
        num //= 1000
        if chunk > 0:
            words = answer_three_digits_exercise(chunk)
            if scales[scale_index]:
                words += " " + scales[scale_index]
            parts.append(words)
        scale_index += 1
    return " ".join(reversed(parts))


for num, expected in tests:
    assert answer_number_to_words_exercise(num) == expected, num


# ===== Đáp án Bài 5 =====
def answer_thousand_separator(n: int) -> str:
    groups = []
    while n >= 1000:
        groups.append(f"{n % 1000:03d}")    # nhóm phía sau: đủ 3 chữ số
        n //= 1000
    groups.append(str(n))                   # nhóm đầu: viết bình thường
    return ".".join(reversed(groups))


assert answer_thousand_separator(987) == "987"
assert answer_thousand_separator(1234) == "1.234"
assert answer_thousand_separator(123456789) == "123.456.789"
assert answer_thousand_separator(0) == "0"
assert answer_thousand_separator(1000010) == "1.000.010"
random.seed(1556)
for _ in range(2000):
    n = random.randint(0, 2**31 - 1)
    assert answer_thousand_separator(n) == f"{n:,}".replace(",", "."), n


# ===== Đáp án Bài 6 =====
def answer_vi_group(g: int, full: bool) -> str:
    h, rest = divmod(g, 100)
    t, u = divmod(rest, 10)
    words = []
    if full or h > 0:
        words += [VI_DIGITS[h], "trăm"]
    if t == 0:
        if u > 0:
            if words:                       # đã có phần "trăm" -> cần "linh"
                words.append("linh")
            words.append(VI_DIGITS[u])
    elif t == 1:
        words.append("mười")
        if u == 5:
            words.append("lăm")
        elif u > 0:
            words.append(VI_DIGITS[u])
    else:
        words += [VI_DIGITS[t], "mươi"]
        if u == 1:
            words.append("mốt")
        elif u == 5:
            words.append("lăm")
        elif u > 0:
            words.append(VI_DIGITS[u])
    return " ".join(words)


def answer_vietnamese_words(num: int) -> str:
    if num == 0:
        return "không"
    groups = split_groups(num)
    top = len(groups) - 1                   # chỉ số của nhóm đầu tiên (cao nhất)
    parts = []
    for i in range(top, -1, -1):
        if groups[i] == 0:
            continue
        text = answer_vi_group(groups[i], full=(i != top))
        if VI_SCALES[i]:
            text += " " + VI_SCALES[i]
        parts.append(text)
    return " ".join(parts)


vi_tests = [
    (0, "không"), (5, "năm"), (10, "mười"), (11, "mười một"), (15, "mười lăm"),
    (20, "hai mươi"), (21, "hai mươi mốt"), (24, "hai mươi bốn"), (25, "hai mươi lăm"),
    (101, "một trăm linh một"), (105, "một trăm linh năm"), (110, "một trăm mười"),
    (1000, "một nghìn"), (1005, "một nghìn không trăm linh năm"),
    (1050, "một nghìn không trăm năm mươi"), (12345, "mười hai nghìn ba trăm bốn mươi lăm"),
    (1000000, "một triệu"), (1000010, "một triệu không trăm mười"),
    (2000001, "hai triệu không trăm linh một"),
    (1234567, "một triệu hai trăm ba mươi bốn nghìn năm trăm sáu mươi bảy"),
    (2147483647, "hai tỷ một trăm bốn mươi bảy triệu bốn trăm tám mươi ba nghìn "
                 "sáu trăm bốn mươi bảy"),
]
for num, expected in vi_tests:
    assert answer_vietnamese_words(num) == expected, (num, answer_vietnamese_words(num))

print("Đáp án Bài 3–6: PASS toàn bộ test!")
for n in [1000010, 2147483647]:
    print(f"{n:>10} -> {answer_vietnamese_words(n)}")

Đáp án Bài 3–6: PASS toàn bộ test!
   1000010 -> một triệu không trăm mười
2147483647 -> hai tỷ một trăm bốn mươi bảy triệu bốn trăm tám mươi ba nghìn sáu trăm bốn mươi bảy


## 21. Template cuối cùng nên nhớ

```python
class Solution:
    ONES = ["", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
    TEENS = ["Ten", "Eleven", "Twelve", "Thirteen", "Fourteen",
             "Fifteen", "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
    TENS = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]
    SCALES = ["", "Thousand", "Million", "Billion"]

    def numberToWords(self, num: int) -> str:
        if num == 0:
            return "Zero"                              # 1) trường hợp đặc biệt
        parts, scale_index = [], 0
        while num > 0:                                 # 2) bóc nhóm 3 chữ số
            chunk = num % 1000
            num //= 1000
            if chunk > 0:                              # 3) bỏ nhóm rỗng
                words = self.three_digits(chunk)
                if self.SCALES[scale_index]:
                    words += " " + self.SCALES[scale_index]
                parts.append(words)
            scale_index += 1                           # 4) luôn tăng cấp
        return " ".join(reversed(parts))               # 5) đảo + join

    def three_digits(self, n: int) -> str:             # 0 <= n < 1000, 0 -> ""
        hundreds, rest = divmod(n, 100)
        parts = []
        if hundreds:
            parts.append(self.ONES[hundreds] + " Hundred")
        if 1 <= rest <= 9:
            parts.append(self.ONES[rest])
        elif 10 <= rest <= 19:
            parts.append(self.TEENS[rest - 10])
        elif rest >= 20:
            parts.append(self.TENS[rest // 10])
            if rest % 10:
                parts.append(self.ONES[rest % 10])
        return " ".join(parts)
```

Câu thần chú: **"Bảng tra cho số nhỏ — nhóm 3 cho số lớn — 0 thì im lặng — join thay cho cộng chuỗi."**

## 22. Checklist tự kiểm tra trước khi submit

- [ ] `num = 0` trả về `"Zero"` chưa? (và helper **không** trả `"Zero"` cho chữ số 0 bên trong)
- [ ] Các số `10`–`19` dùng bảng `TEENS` chưa? (`15` → `Fifteen`, không phải `Ten Five`)
- [ ] Bội của 10 (`20`, `40`, `90`) không bị thêm từ thừa hay dấu cách cuối?
- [ ] Bội của 100 (`100`, `500`) chỉ là `"X Hundred"`?
- [ ] Nhóm rỗng ở giữa (`1000010`, `1000000`) không in tên cấp?
- [ ] `scale_index` tăng ở **mọi** vòng, kể cả nhóm rỗng?
- [ ] Không có dấu cách kép / đầu / cuối (thử in bằng `repr`)?
- [ ] Chính tả: `Forty`, `Fifty`, `Eighteen`, `Ninety`, `Twelve`, `Nineteen`?
- [ ] Không có `"and"`, gạch nối, chữ thường?
- [ ] Đã thử giá trị lớn nhất `2147483647` và ba ví dụ của đề?